In [ ]:
# ============================================================
# DATASET EXPLORATION SCRIPT
# Purpose: Extract comprehensive information about the dataset
# ============================================================

import os
import pandas as pd
import numpy as np
from PIL import Image
import json
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# CONFIGURATION - Update this path to match your Kaggle dataset
# ============================================================
# Common Kaggle dataset paths - try to find the correct one
possible_paths = [
    '/kaggle/input/afssssssasf/',
    '/kaggle/input/breast-cancer-dataset/',
    '/kaggle/input/breast-cancer-histopathology/',
    '/kaggle/input/brca-dataset/',
]

# Find the actual dataset path
DATASET_PATH = None
for path in possible_paths:
    if os.path.exists(path):
        contents = os.listdir(path)
        print(f"Found path: {path}")
        print(f"Contents: {contents[:10]}...")  # First 10 items
        DATASET_PATH = path
        break

if DATASET_PATH is None:
    DATASET_PATH = '/kaggle/input/afssssssasf/'

print(f"\n{'='*60}")
print("DATASET EXPLORATION REPORT")
print(f"{'='*60}\n")

# ============================================================
# 1. EXPLORE TOP-LEVEL STRUCTURE
# ============================================================
print("=" * 60)
print("1. TOP-LEVEL DIRECTORY STRUCTURE")
print("=" * 60)

def explore_directory(path, max_depth=3, current_depth=0, max_items=20):
    """Recursively explore directory structure"""
    result = []
    try:
        items = sorted(os.listdir(path))
        for i, item in enumerate(items[:max_items]):
            item_path = os.path.join(path, item)
            prefix = "  " * current_depth + "├── " if i < len(items[:max_items]) - 1 else "  " * current_depth + "└── "

            if os.path.isdir(item_path):
                result.append(f"{prefix}📁 {item}/")
                if current_depth < max_depth - 1:
                    result.extend(explore_directory(item_path, max_depth, current_depth + 1, max_items=5))
            else:
                size = os.path.getsize(item_path)
                size_str = f"{size/1024:.1f}KB" if size < 1024*1024 else f"{size/(1024*1024):.1f}MB"
                result.append(f"{prefix}📄 {item} ({size_str})")

        if len(items) > max_items:
            result.append(f"{'  ' * current_depth}    ... and {len(items) - max_items} more items")
    except Exception as e:
        result.append(f"Error: {e}")
    return result

# List all items in input directory
print(f"\nExploring: {DATASET_PATH}")
all_items = os.listdir(DATASET_PATH)
print(f"Total items in input: {len(all_items)}")
print("\nDirectory tree (first 3 levels):")
tree = explore_directory(DATASET_PATH, max_depth=3, max_items=15)
for line in tree:
    print(line)

# ============================================================
# 2. FIND ALL TCGA FOLDERS AND ANALYZE STRUCTURE
# ============================================================
print("\n" + "=" * 60)
print("2. TCGA FOLDERS ANALYSIS")
print("=" * 60)

def find_all_tcga_folders(base_path):
    """Find all TCGA-* folders recursively"""
    tcga_folders = []
    for root, dirs, files in os.walk(base_path):
        for d in dirs:
            if d.startswith('TCGA-'):
                tcga_folders.append(os.path.join(root, d))
    return tcga_folders

tcga_folders = find_all_tcga_folders(DATASET_PATH)
print(f"\nTotal TCGA folders found: {len(tcga_folders)}")

if tcga_folders:
    print("\nFirst 10 TCGA folder names:")
    for folder in tcga_folders[:10]:
        print(f"  - {os.path.basename(folder)}")
    if len(tcga_folders) > 10:
        print(f"  ... and {len(tcga_folders) - 10} more")

# ============================================================
# 3. ANALYZE FOLDER CONTENTS (Images + CSVs)
# ============================================================
print("\n" + "=" * 60)
print("3. FOLDER CONTENTS ANALYSIS")
print("=" * 60)

folder_stats = []
sample_folder = None

for folder in tcga_folders[:50]:  # Analyze first 50 folders
    folder_name = os.path.basename(folder)
    contents = os.listdir(folder)

    # Count files by type
    jpg_files = [f for f in contents if f.endswith('.jpg') or f.endswith('.jpeg')]
    png_files = [f for f in contents if f.endswith('.png')]
    csv_files = [f for f in contents if f.endswith('.csv')]
    other_files = [f for f in contents if not any(f.endswith(ext) for ext in ['.jpg', '.jpeg', '.png', '.csv'])]

    folder_stats.append({
        'folder': folder_name,
        'path': folder,
        'jpg_count': len(jpg_files),
        'png_count': len(png_files),
        'csv_count': len(csv_files),
        'other_count': len(other_files),
        'total_files': len(contents),
        'csv_files': csv_files
    })

    if sample_folder is None and len(jpg_files) > 0:
        sample_folder = folder

# Summary statistics
df_stats = pd.DataFrame(folder_stats)
print(f"\nAnalyzed {len(folder_stats)} folders:")
print(f"  - Total JPG images: {df_stats['jpg_count'].sum()}")
print(f"  - Total PNG images: {df_stats['png_count'].sum()}")
print(f"  - Total CSV files: {df_stats['csv_count'].sum()}")
print(f"  - Avg images per folder: {df_stats['jpg_count'].mean():.1f}")
print(f"  - Min images in a folder: {df_stats['jpg_count'].min()}")
print(f"  - Max images in a folder: {df_stats['jpg_count'].max()}")

print("\nImage count distribution:")
print(df_stats['jpg_count'].describe())

# Show folders with most images
print("\nTop 5 folders by image count:")
top_folders = df_stats.nlargest(5, 'jpg_count')[['folder', 'jpg_count', 'csv_count']]
print(top_folders.to_string(index=False))

# ============================================================
# 4. ANALYZE SAMPLE IMAGES
# ============================================================
print("\n" + "=" * 60)
print("4. IMAGE ANALYSIS")
print("=" * 60)

if sample_folder:
    print(f"\nAnalyzing images from: {os.path.basename(sample_folder)}")

    image_files = sorted([f for f in os.listdir(sample_folder) if f.endswith(('.jpg', '.jpeg', '.png'))])

    print(f"Total images in sample folder: {len(image_files)}")
    print(f"First 10 image names: {image_files[:10]}")
    print(f"Last 5 image names: {image_files[-5:]}")

    # Analyze a few sample images
    image_info = []
    sample_images = image_files[:20]  # Analyze first 20 images

    print("\nImage properties (first 20 images):")
    for img_name in sample_images:
        img_path = os.path.join(sample_folder, img_name)
        try:
            with Image.open(img_path) as img:
                image_info.append({
                    'name': img_name,
                    'width': img.size[0],
                    'height': img.size[1],
                    'mode': img.mode,
                    'format': img.format,
                    'size_kb': os.path.getsize(img_path) / 1024
                })
        except Exception as e:
            print(f"  Error reading {img_name}: {e}")

    if image_info:
        df_images = pd.DataFrame(image_info)
        print(df_images.to_string(index=False))

        print("\nImage dimension statistics:")
        print(f"  Width  - min: {df_images['width'].min()}, max: {df_images['width'].max()}, unique: {df_images['width'].nunique()}")
        print(f"  Height - min: {df_images['height'].min()}, max: {df_images['height'].max()}, unique: {df_images['height'].nunique()}")
        print(f"  Modes: {df_images['mode'].unique().tolist()}")
        print(f"  Avg file size: {df_images['size_kb'].mean():.1f} KB")

        # Check if all images have same dimensions
        if df_images['width'].nunique() == 1 and df_images['height'].nunique() == 1:
            print(f"\n✓ All images have SAME dimensions: {df_images['width'].iloc[0]}x{df_images['height'].iloc[0]}")
        else:
            print("\n⚠ Images have VARYING dimensions!")

# ============================================================
# 5. ANALYZE CSV FILES (patch_info.csv)
# ============================================================
print("\n" + "=" * 60)
print("5. CSV FILE ANALYSIS (patch_info.csv)")
print("=" * 60)

csv_found = False
sample_csv_path = None

# Find a CSV file
for stat in folder_stats:
    if stat['csv_files']:
        for csv_name in stat['csv_files']:
            sample_csv_path = os.path.join(stat['path'], csv_name)
            if os.path.exists(sample_csv_path):
                csv_found = True
                break
    if csv_found:
        break

if csv_found and sample_csv_path:
    print(f"\nAnalyzing: {sample_csv_path}")

    try:
        # Read CSV
        df_csv = pd.read_csv(sample_csv_path)

        print(f"\n--- CSV Basic Info ---")
        print(f"Shape: {df_csv.shape[0]} rows × {df_csv.shape[1]} columns")
        print(f"Memory usage: {df_csv.memory_usage(deep=True).sum() / (1024*1024):.2f} MB")

        print(f"\n--- Column Names ({len(df_csv.columns)} total) ---")
        columns = df_csv.columns.tolist()

        # Group columns by prefix
        r_cols = [c for c in columns if c.startswith('R') and c[1:].isdigit()]
        g_cols = [c for c in columns if c.startswith('G') and c[1:].isdigit()]
        b_cols = [c for c in columns if c.startswith('B') and c[1:].isdigit()]
        other_cols = [c for c in columns if c not in r_cols + g_cols + b_cols]

        print(f"\nOther columns: {other_cols}")
        print(f"R columns: {len(r_cols)} (R0 to R{r_cols[-1][1:] if r_cols else 'N/A'})")
        print(f"G columns: {len(g_cols)} (G0 to G{g_cols[-1][1:] if g_cols else 'N/A'})")
        print(f"B columns: {len(b_cols)} (B0 to B{b_cols[-1][1:] if b_cols else 'N/A'})")

        print(f"\n--- Data Types ---")
        print(df_csv.dtypes.value_counts())

        print(f"\n--- First 5 Rows (non-RGB columns) ---")
        if other_cols:
            print(df_csv[other_cols].head().to_string())

        print(f"\n--- Sample of 'id' column ---")
        if 'id' in df_csv.columns:
            print(f"Unique values: {df_csv['id'].nunique()}")
            print(f"Sample values: {df_csv['id'].head(10).tolist()}")
            print(f"Data type: {df_csv['id'].dtype}")

        print(f"\n--- Sample of 'cell index' column ---")
        cell_idx_col = None
        for col in other_cols:
            if 'cell' in col.lower() or 'index' in col.lower():
                cell_idx_col = col
                break

        if cell_idx_col:
            print(f"Column name: '{cell_idx_col}'")
            print(f"Unique values: {df_csv[cell_idx_col].nunique()}")
            print(f"Statistics: min={df_csv[cell_idx_col].min():.2f}, max={df_csv[cell_idx_col].max():.2f}, mean={df_csv[cell_idx_col].mean():.2f}")
            print(f"Sample values: {df_csv[cell_idx_col].head(10).tolist()}")

        print(f"\n--- RGB Histogram Data Analysis ---")
        print("(These appear to be color histograms with 255 bins each)")

        # Analyze R columns
        if r_cols:
            r_data = df_csv[r_cols]
            print(f"\nR channel histogram statistics (across all rows):")
            print(f"  Sum of R0-R50 (darker pixels): {r_data[r_cols[:51]].sum().sum():.0f}")
            print(f"  Sum of R100-R150 (mid pixels): {r_data[r_cols[100:151]].sum().sum():.0f}")
            print(f"  Sum of R200-R254 (bright pixels): {r_data[r_cols[200:]].sum().sum():.0f}")

            # Single row example
            print(f"\n  Example row 0 - R histogram:")
            print(f"    R0-R10: {r_data.iloc[0, :11].tolist()}")
            print(f"    R120-R130: {r_data.iloc[0, 120:131].tolist()}")
            print(f"    R244-R254: {r_data.iloc[0, -11:].tolist()}")

        # Check for any other useful columns
        print(f"\n--- Checking for additional meaningful columns ---")
        for col in other_cols:
            if col not in ['id']:
                print(f"\n{col}:")
                print(f"  Type: {df_csv[col].dtype}")
                print(f"  Null count: {df_csv[col].isnull().sum()}")
                print(f"  Sample: {df_csv[col].head(5).tolist()}")
                if pd.api.types.is_numeric_dtype(df_csv[col]):
                    print(f"  Stats: min={df_csv[col].min()}, max={df_csv[col].max()}, mean={df_csv[col].mean():.2f}")

    except Exception as e:
        print(f"Error reading CSV: {e}")
        import traceback
        traceback.print_exc()
else:
    print("No CSV files found!")

# ============================================================
# 6. VERIFY RELATIONSHIP BETWEEN IMAGES AND CSV
# ============================================================
print("\n" + "=" * 60)
print("6. IMAGE-CSV RELATIONSHIP VERIFICATION")
print("=" * 60)

if sample_folder and csv_found:
    folder_images = sorted([f for f in os.listdir(sample_folder) if f.endswith(('.jpg', '.jpeg', '.png'))])

    # Get image IDs from filenames
    image_ids = []
    for img in folder_images[:20]:
        # Extract number from filename (e.g., "0.jpg" -> 0)
        try:
            img_id = int(os.path.splitext(img)[0])
            image_ids.append(img_id)
        except:
            image_ids.append(img)

    print(f"First 20 image IDs from filenames: {image_ids}")

    if 'id' in df_csv.columns:
        csv_ids = df_csv['id'].head(20).tolist()
        print(f"First 20 IDs from CSV: {csv_ids}")

        # Check alignment
        if len(image_ids) > 0:
            print(f"\nDo image IDs (as int) match CSV IDs?")
            for i in range(min(5, len(image_ids))):
                img_id = image_ids[i]
                csv_id = csv_ids[i] if i < len(csv_ids) else None
                match = "✓" if (img_id == csv_id or img_id == int(csv_id) if csv_id else False) else "✗"
                print(f"  Image {i}: {img_id} vs CSV: {csv_id} {match}")

# ============================================================
# 7. ANALYZE MULTIPLE FOLDERS FOR CONSISTENCY
# ============================================================
print("\n" + "=" * 60)
print("7. CROSS-FOLDER CONSISTENCY CHECK")
print("=" * 60)

# Check if CSV structure is consistent across folders
csv_structures = []
for stat in folder_stats[:10]:  # Check first 10 folders
    if stat['csv_files']:
        csv_path = os.path.join(stat['path'], stat['csv_files'][0])
        try:
            df = pd.read_csv(csv_path, nrows=1)
            csv_structures.append({
                'folder': stat['folder'],
                'num_columns': len(df.columns),
                'columns_hash': hash(tuple(df.columns.tolist()))
            })
        except:
            pass

if csv_structures:
    print("\nCSV structure comparison:")
    for s in csv_structures[:5]:
        print(f"  {s['folder']}: {s['num_columns']} columns")

    # Check if all have same structure
    unique_structures = len(set(s['columns_hash'] for s in csv_structures))
    if unique_structures == 1:
        print(f"\n✓ All checked CSVs have IDENTICAL column structure")
    else:
        print(f"\n⚠ Found {unique_structures} different column structures!")

# ============================================================
# 8. DISK SPACE AND MEMORY ESTIMATION
# ============================================================
print("\n" + "=" * 60)
print("8. RESOURCE ESTIMATION")
print("=" * 60)

# Estimate total dataset size
total_images = df_stats['jpg_count'].sum() + df_stats['png_count'].sum()
avg_image_size_kb = df_images['size_kb'].mean() if 'df_images' in dir() else 50  # Default estimate

print(f"\nEstimated Statistics:")
print(f"  Total folders: {len(tcga_folders)}")
print(f"  Total images (from analyzed folders): {total_images}")
print(f"  Estimated total images (all folders): {total_images * len(tcga_folders) / len(folder_stats):.0f}")
print(f"  Avg image size: {avg_image_size_kb:.1f} KB")

# Memory estimation for training
if 'df_images' in dir() and len(df_images) > 0:
    img_w = df_images['width'].iloc[0]
    img_h = df_images['height'].iloc[0]
    channels = 3 if df_images['mode'].iloc[0] == 'RGB' else 1

    single_image_memory_mb = (img_w * img_h * channels * 4) / (1024 * 1024)  # float32
    batch_32_memory_mb = single_image_memory_mb * 32

    print(f"\nMemory Requirements (for training):")
    print(f"  Single image ({img_w}x{img_h}x{channels}): {single_image_memory_mb:.2f} MB (float32)")
    print(f"  Batch of 32: {batch_32_memory_mb:.2f} MB")
    print(f"  Batch of 16: {batch_32_memory_mb/2:.2f} MB")
    print(f"  Batch of 8: {batch_32_memory_mb/4:.2f} MB")

# ============================================================
# 9. SUMMARY FOR AI ASSISTANT
# ============================================================
print("\n" + "=" * 60)
print("9. SUMMARY FOR AI ASSISTANT")
print("=" * 60)

summary = f"""
=== DATASET SUMMARY ===

STRUCTURE:
- Dataset path: {DATASET_PATH}
- Number of TCGA patient folders: {len(tcga_folders)}
- Total images (analyzed): {total_images}
- Avg images per folder: {df_stats['jpg_count'].mean():.1f}
- Image range per folder: {df_stats['jpg_count'].min()} to {df_stats['jpg_count'].max()}

IMAGE PROPERTIES:
"""

if 'df_images' in dir() and len(df_images) > 0:
    summary += f"""- Dimensions: {df_images['width'].iloc[0]}x{df_images['height'].iloc[0]}
- Color mode: {df_images['mode'].iloc[0]}
- Format: {df_images['format'].iloc[0]}
- Avg file size: {df_images['size_kb'].mean():.1f} KB
- Consistent dimensions: {'Yes' if df_images['width'].nunique() == 1 else 'No'}
"""

summary += f"""
CSV STRUCTURE:
- File name: patch_info.csv
- Rows per file: ~{df_csv.shape[0] if 'df_csv' in dir() else 'Unknown'}
- Total columns: {df_csv.shape[1] if 'df_csv' in dir() else 'Unknown'}
- Non-RGB columns: {other_cols if 'other_cols' in dir() else 'Unknown'}
- R columns: {len(r_cols) if 'r_cols' in dir() else 0} (R0 to R254 - RED histogram)
- G columns: {len(g_cols) if 'g_cols' in dir() else 0} (G0 to G254 - GREEN histogram)
- B columns: {len(b_cols) if 'b_cols' in dir() else 0} (B0 to B254 - BLUE histogram)

DATA INTERPRETATION:
- The R/G/B columns appear to be COLOR HISTOGRAMS (256 bins each)
- Each row likely corresponds to one image patch
- 'id' column maps to image filename (0.jpg, 1.jpg, etc.)
- 'cell index' might be a cell density or count metric

READY FOR HOVERNET + GNN PIPELINE:
- Images: Ready for segmentation
- Cell data: Available via 'cell index' column
- Graph construction: Can use cell positions from segmentation
- Histograms: Can be used as node features in GNN
"""

print(summary)

# ============================================================
# 10. SAVE REPORT TO FILE
# ============================================================
print("\n" + "=" * 60)
print("10. SAVING REPORT")
print("=" * 60)

report_path = '/kaggle/working/dataset_exploration_report.txt'
with open(report_path, 'w') as f:
    f.write(summary)
print(f"Report saved to: {report_path}")

print("\n" + "=" * 60)
print("EXPLORATION COMPLETE!")
print("=" * 60)
print("\nCopy all the output above and share it with the AI assistant")
print("for customized code generation for your project.")

In [ ]:
Found path: /kaggle/input/afssssssasf/
Contents: ['TCGA-AO-A1KR', 'TCGA-A2-A0D4', 'TCGA-A7-A4SB', 'TCGA-A2-A0CR', 'TCGA-AC-A7VB', 'TCGA-A7-A4SD', 'TCGA-A8-A07S', 'TCGA-A8-A0A4', 'TCGA-A7-A0CJ', 'TCGA-AO-A1KO']...

============================================================
DATASET EXPLORATION REPORT
============================================================

============================================================
1. TOP-LEVEL DIRECTORY STRUCTURE
============================================================

Exploring: /kaggle/input/afssssssasf/
Total items in input: 299

Directory tree (first 3 levels):
├── 📁 TCGA-3C-AALI/
  ├── 📁 .ipynb_checkpoints/
    ├── 📄 0-checkpoint.jpg (240.0KB)
    ├── 📄 1-checkpoint.jpg (259.1KB)
    ├── 📄 10-checkpoint.jpg (215.9KB)
    ├── 📄 13-checkpoint.jpg (249.9KB)
    └── 📄 15-checkpoint.jpg (224.6KB)
        ... and 6 more items
  ├── 📄 0.jpg (240.0KB)
  ├── 📄 1.jpg (259.1KB)
  ├── 📄 10.jpg (215.9KB)
  └── 📄 100.jpg (293.2KB)
      ... and 606 more items
├── 📁 TCGA-3C-AALJ/
  ├── 📄 0.jpg (251.8KB)
  ├── 📄 1.jpg (264.0KB)
  ├── 📄 10.jpg (266.7KB)
  ├── 📄 100.jpg (289.1KB)
  └── 📄 101.jpg (282.1KB)
      ... and 926 more items
├── 📁 TCGA-3C-AALK/
  ├── 📄 0.jpg (213.7KB)
  ├── 📄 1.jpg (252.6KB)
  ├── 📄 10.jpg (208.7KB)
  ├── 📄 100.jpg (178.6KB)
  └── 📄 101.jpg (196.3KB)
      ... and 582 more items
├── 📁 TCGA-4H-AAAK/
  ├── 📄 0.jpg (263.0KB)
  ├── 📄 1.jpg (283.2KB)
  ├── 📄 10.jpg (275.5KB)
  ├── 📄 100.jpg (303.6KB)
  └── 📄 101.jpg (291.6KB)
      ... and 601 more items
├── 📁 TCGA-5T-A9QA/
  ├── 📄 0.jpg (264.7KB)
  ├── 📄 1.jpg (276.5KB)
  ├── 📄 10.jpg (287.2KB)
  ├── 📄 100.jpg (270.8KB)
  └── 📄 1000.jpg (257.1KB)
      ... and 1283 more items
├── 📁 TCGA-A1-A0SB/
  ├── 📄 0.jpg (167.8KB)
  ├── 📄 1.jpg (173.7KB)
  ├── 📄 10.jpg (182.7KB)
  ├── 📄 100.jpg (199.0KB)
  └── 📄 101.jpg (198.2KB)
      ... and 201 more items
├── 📁 TCGA-A1-A0SD/
  ├── 📄 0.jpg (205.6KB)
  ├── 📄 1.jpg (210.2KB)
  ├── 📄 10.jpg (186.3KB)
  ├── 📄 100.jpg (169.8KB)
  └── 📄 101.jpg (185.1KB)
      ... and 182 more items
├── 📁 TCGA-A1-A0SE/
  ├── 📄 0.jpg (196.0KB)
  ├── 📄 1.jpg (199.7KB)
  ├── 📄 10.jpg (173.0KB)
  ├── 📄 100.jpg (73.9KB)
  └── 📄 1000.jpg (170.3KB)
      ... and 1035 more items
├── 📁 TCGA-A1-A0SF/
  ├── 📄 0.jpg (138.4KB)
  ├── 📄 1.jpg (172.1KB)
  ├── 📄 10.jpg (224.2KB)
  ├── 📄 100.jpg (179.7KB)
  └── 📄 101.jpg (175.1KB)
      ... and 373 more items
├── 📁 TCGA-A1-A0SH/
  ├── 📄 0.jpg (167.8KB)
  ├── 📄 1.jpg (201.9KB)
  ├── 📄 10.jpg (196.6KB)
  ├── 📄 100.jpg (186.9KB)
  └── 📄 101.jpg (191.7KB)
      ... and 413 more items
├── 📁 TCGA-A1-A0SI/
  ├── 📄 0.jpg (188.0KB)
  ├── 📄 1.jpg (195.9KB)
  ├── 📄 10.jpg (231.7KB)
  ├── 📄 100.jpg (241.3KB)
  └── 📄 101.jpg (238.4KB)
      ... and 412 more items
├── 📁 TCGA-A1-A0SJ/
  ├── 📄 0.jpg (213.4KB)
  ├── 📄 1.jpg (226.0KB)
  ├── 📄 10.jpg (207.9KB)
  ├── 📄 100.jpg (211.9KB)
  └── 📄 101.jpg (189.9KB)
      ... and 261 more items
├── 📁 TCGA-A1-A0SK/
  ├── 📄 0.jpg (235.1KB)
  ├── 📄 1.jpg (223.9KB)
  ├── 📄 10.jpg (187.6KB)
  ├── 📄 100.jpg (222.8KB)
  └── 📄 1000.jpg (187.9KB)
      ... and 1049 more items
├── 📁 TCGA-A1-A0SM/
  ├── 📄 0.jpg (173.9KB)
  ├── 📄 1.jpg (198.6KB)
  ├── 📄 10.jpg (181.7KB)
  ├── 📄 100.jpg (206.6KB)
  └── 📄 101.jpg (208.4KB)
      ... and 470 more items
└── 📁 TCGA-A1-A0SN/
  ├── 📄 0.jpg (196.3KB)
  ├── 📄 1.jpg (225.3KB)
  ├── 📄 10.jpg (184.3KB)
  ├── 📄 100.jpg (203.2KB)
  └── 📄 101.jpg (213.0KB)
      ... and 276 more items
    ... and 284 more items

============================================================
2. TCGA FOLDERS ANALYSIS
============================================================

Total TCGA folders found: 299

First 10 TCGA folder names:
  - TCGA-AO-A1KR
  - TCGA-A2-A0D4
  - TCGA-A7-A4SB
  - TCGA-A2-A0CR
  - TCGA-AC-A7VB
  - TCGA-A7-A4SD
  - TCGA-A8-A07S
  - TCGA-A8-A0A4
  - TCGA-A7-A0CJ
  - TCGA-AO-A1KO
  ... and 289 more

============================================================
3. FOLDER CONTENTS ANALYSIS
============================================================

Analyzed 50 folders:
  - Total JPG images: 19642
  - Total PNG images: 0
  - Total CSV files: 50
  - Avg images per folder: 392.8
  - Min images in a folder: 55
  - Max images in a folder: 1372

Image count distribution:
count      50.000000
mean      392.840000
std       365.165904
min        55.000000
25%        95.500000
50%       242.000000
75%       570.500000
max      1372.000000
Name: jpg_count, dtype: float64

Top 5 folders by image count:
      folder  jpg_count  csv_count
TCGA-AO-A1KR       1372          1
TCGA-5T-A9QA       1287          1
TCGA-A2-A0SY       1143          1
TCGA-AO-A0J6       1132          1
TCGA-A2-A04V       1053          1

============================================================
4. IMAGE ANALYSIS
============================================================

Analyzing images from: TCGA-AO-A1KR
Total images in sample folder: 1372
First 10 image names: ['0.jpg', '1.jpg', '10.jpg', '100.jpg', '1000.jpg', '1001.jpg', '1002.jpg', '1003.jpg', '1004.jpg', '1005.jpg']
Last 5 image names: ['995.jpg', '996.jpg', '997.jpg', '998.jpg', '999.jpg']

Image properties (first 20 images):
    name  width  height mode format    size_kb
   0.jpg   1000    1000  RGB   JPEG 281.389648
   1.jpg   1000    1000  RGB   JPEG 248.692383
  10.jpg   1000    1000  RGB   JPEG 285.504883
 100.jpg   1000    1000  RGB   JPEG 310.087891
1000.jpg   1000    1000  RGB   JPEG 317.933594
1001.jpg   1000    1000  RGB   JPEG 336.609375
1002.jpg   1000    1000  RGB   JPEG 319.734375
1003.jpg   1000    1000  RGB   JPEG 323.373047
1004.jpg   1000    1000  RGB   JPEG 311.711914
1005.jpg   1000    1000  RGB   JPEG 337.901367
1006.jpg   1000    1000  RGB   JPEG 318.157227
1007.jpg   1000    1000  RGB   JPEG 322.779297
1008.jpg   1000    1000  RGB   JPEG 323.899414
1009.jpg   1000    1000  RGB   JPEG 328.965820
 101.jpg   1000    1000  RGB   JPEG 280.375977
1010.jpg   1000    1000  RGB   JPEG 329.574219
1011.jpg   1000    1000  RGB   JPEG 339.740234
1012.jpg   1000    1000  RGB   JPEG 325.144531
1013.jpg   1000    1000  RGB   JPEG 341.265625
1014.jpg   1000    1000  RGB   JPEG 327.902344

Image dimension statistics:
  Width  - min: 1000, max: 1000, unique: 1
  Height - min: 1000, max: 1000, unique: 1
  Modes: ['RGB']
  Avg file size: 315.5 KB

✓ All images have SAME dimensions: 1000x1000

============================================================
5. CSV FILE ANALYSIS (patch_info.csv)
============================================================

Analyzing: /kaggle/input/afssssssasf/TCGA-AO-A1KR/patch_info.csv

--- CSV Basic Info ---
Shape: 1372 rows × 767 columns
Memory usage: 8.03 MB

--- Column Names (767 total) ---

Other columns: ['ID', 'cell index']
R columns: 255 (R0 to R254)
G columns: 255 (G0 to G254)
B columns: 255 (B0 to B254)

--- Data Types ---
float64    767
Name: count, dtype: int64

--- First 5 Rows (non-RGB columns) ---
    ID  cell index
0  0.0   42.644044
1  3.0   40.979924
2  1.0   30.624391
3  2.0   29.056965
4  4.0   29.487002

--- Sample of 'id' column ---

--- Sample of 'cell index' column ---
Column name: 'cell index'
Unique values: 1372
Statistics: min=22.93, max=77.18, mean=45.06
Sample values: [42.64404441813223, 40.97992352322207, 30.624391110488403, 29.056964535110268, 29.487002369751437, 30.37609834201166, 35.973951030466644, 32.80538338265421, 30.27175742416209, 26.19178351492376]

--- RGB Histogram Data Analysis ---
(These appear to be color histograms with 255 bins each)

R channel histogram statistics (across all rows):
  Sum of R0-R50 (darker pixels): 34647227
  Sum of R100-R150 (mid pixels): 315677633
  Sum of R200-R254 (bright pixels): 404668081

  Example row 0 - R histogram:
    R0-R10: [60.0, 24.0, 21.0, 22.0, 24.0, 47.0, 30.0, 61.0, 62.0, 74.0, 107.0]
    R120-R130: [2800.0, 2929.0, 3025.0, 3167.0, 3193.0, 3290.0, 3359.0, 3482.0, 3611.0, 3576.0, 3718.0]
    R244-R254: [504.0, 419.0, 339.0, 318.0, 264.0, 199.0, 200.0, 178.0, 141.0, 111.0, 485.0]

--- Checking for additional meaningful columns ---

ID:
  Type: float64
  Null count: 0
  Sample: [0.0, 3.0, 1.0, 2.0, 4.0]
  Stats: min=0.0, max=1371.0, mean=685.50

cell index:
  Type: float64
  Null count: 0
  Sample: [42.64404441813223, 40.97992352322207, 30.624391110488403, 29.056964535110268, 29.487002369751437]
  Stats: min=22.934730503729494, max=77.1783366502608, mean=45.06

============================================================
6. IMAGE-CSV RELATIONSHIP VERIFICATION
============================================================
First 20 image IDs from filenames: [0, 1, 10, 100, 1000, 1001, 1002, 1003, 1004, 1005, 1006, 1007, 1008, 1009, 101, 1010, 1011, 1012, 1013, 1014]

============================================================
7. CROSS-FOLDER CONSISTENCY CHECK
============================================================

CSV structure comparison:
  TCGA-AO-A1KR: 767 columns
  TCGA-A2-A0D4: 767 columns
  TCGA-A7-A4SB: 767 columns
  TCGA-A2-A0CR: 767 columns
  TCGA-AC-A7VB: 767 columns

✓ All checked CSVs have IDENTICAL column structure

============================================================
8. RESOURCE ESTIMATION
============================================================

Estimated Statistics:
  Total folders: 299
  Total images (from analyzed folders): 19642
  Estimated total images (all folders): 117459
  Avg image size: 315.5 KB

Memory Requirements (for training):
  Single image (1000x1000x3): 11.44 MB (float32)
  Batch of 32: 366.21 MB
  Batch of 16: 183.11 MB
  Batch of 8: 91.55 MB

============================================================
9. SUMMARY FOR AI ASSISTANT
============================================================

=== DATASET SUMMARY ===

STRUCTURE:
- Dataset path: /kaggle/input/afssssssasf/
- Number of TCGA patient folders: 299
- Total images (analyzed): 19642
- Avg images per folder: 392.8
- Image range per folder: 55 to 1372

IMAGE PROPERTIES:
- Dimensions: 1000x1000
- Color mode: RGB
- Format: JPEG
- Avg file size: 315.5 KB
- Consistent dimensions: Yes

CSV STRUCTURE:
- File name: patch_info.csv
- Rows per file: ~1372
- Total columns: 767
- Non-RGB columns: ['ID', 'cell index']
- R columns: 255 (R0 to R254 - RED histogram)
- G columns: 255 (G0 to G254 - GREEN histogram)
- B columns: 255 (B0 to B254 - BLUE histogram)

DATA INTERPRETATION:
- The R/G/B columns appear to be COLOR HISTOGRAMS (256 bins each)
- Each row likely corresponds to one image patch
- 'id' column maps to image filename (0.jpg, 1.jpg, etc.)
- 'cell index' might be a cell density or count metric

READY FOR HOVERNET + GNN PIPELINE:
- Images: Ready for segmentation
- Cell data: Available via 'cell index' column
- Graph construction: Can use cell positions from segmentation
- Histograms: Can be used as node features in GNN


============================================================
10. SAVING REPORT
============================================================
Report saved to: /kaggle/working/dataset_exploration_report.txt

============================================================
EXPLORATION COMPLETE!
============================================================

Copy all the output above and share it with the AI assistant
for customized code generation for your project.

In [1]:
# ============================================================
# CELL 1: MINIMAL SETUP FOR HOVERNET-GNN FIX ONLY
# ============================================================

import os
import gc
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm
import json

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.cuda.amp import GradScaler, autocast

import torchvision.transforms as transforms
from torchvision import models

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

import cv2
import warnings
warnings.filterwarnings('ignore')

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Config - Same as before
class Config:
    DATASET_PATH = '/kaggle/input/datasets/prathameshanvekar/afssssssasf'
    OUTPUT_PATH = '/kaggle/working/'
    NUM_FOLDERS_TO_USE = 40
    MAX_IMAGES_PER_FOLDER = 100
    IMAGE_SIZE = 256
    BATCH_SIZE = 32
    NUM_CLASSES = 3
    USE_AMP = True

config = Config()
os.makedirs(f"{config.OUTPUT_PATH}/models", exist_ok=True)
os.makedirs(f"{config.OUTPUT_PATH}/results", exist_ok=True)

# Class labels
class_labels = ['Low Density', 'Medium Density', 'High Density']

print("✅ Setup complete")

Using device: cuda
✅ Setup complete


In [2]:
# ============================================================
# CELL 2: LOAD DATA AND DEFINE HOVERNET-GNN MODEL
# ============================================================

# ----------------------
# LOAD DATA
# ----------------------
print("Loading data...")

def load_dataset(dataset_path, num_folders=40, max_images_per_folder=100):
    all_data = []
    all_folders = sorted([f for f in os.listdir(dataset_path) 
                         if f.startswith('TCGA-') and os.path.isdir(os.path.join(dataset_path, f))])
    selected_folders = all_folders[:num_folders]
    
    for folder_name in tqdm(selected_folders, desc="Loading folders"):
        folder_path = os.path.join(dataset_path, folder_name)
        csv_path = os.path.join(folder_path, 'patch_info.csv')
        
        if not os.path.exists(csv_path):
            continue
            
        try:
            df = pd.read_csv(csv_path)
        except:
            continue
        
        image_files = sorted([f for f in os.listdir(folder_path) if f.endswith('.jpg')])[:max_images_per_folder]
        
        for img_file in image_files:
            img_id = int(os.path.splitext(img_file)[0])
            row = df[df['ID'] == float(img_id)]
            
            if len(row) == 0:
                continue
            
            row = row.iloc[0]
            all_data.append({
                'image_path': os.path.join(folder_path, img_file),
                'cell_index': row['cell index'],
                'r_hist': row[[f'R{i}' for i in range(0, 255, 25)]].values.astype(np.float32),
                'g_hist': row[[f'G{i}' for i in range(0, 255, 25)]].values.astype(np.float32),
                'b_hist': row[[f'B{i}' for i in range(0, 255, 25)]].values.astype(np.float32)
            })
    
    return pd.DataFrame(all_data)

df_data = load_dataset(config.DATASET_PATH, config.NUM_FOLDERS_TO_USE, config.MAX_IMAGES_PER_FOLDER)
print(f"Loaded {len(df_data)} samples")

# Create labels
cell_indices = df_data['cell_index'].values
thresholds = np.percentile(cell_indices, [33, 66])
df_data['label'] = 0
df_data.loc[df_data['cell_index'] >= thresholds[0], 'label'] = 1
df_data.loc[df_data['cell_index'] >= thresholds[1], 'label'] = 2

# Split
train_df, temp_df = train_test_split(df_data, test_size=0.3, stratify=df_data['label'], random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df['label'], random_state=42)
train_df, val_df, test_df = train_df.reset_index(drop=True), val_df.reset_index(drop=True), test_df.reset_index(drop=True)
print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")

# ----------------------
# TRANSFORMS
# ----------------------
train_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.RandomHorizontalFlip(0.5),
    transforms.RandomVerticalFlip(0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# ----------------------
# HOVERNET ENCODER (Required for the hybrid model)
# ----------------------
class HoVerNetEncoder(nn.Module):
    def __init__(self, pretrained=True):
        super(HoVerNetEncoder, self).__init__()
        resnet = models.resnet34(weights='IMAGENET1K_V1' if pretrained else None)
        self.conv1 = resnet.conv1
        self.bn1 = resnet.bn1
        self.relu = resnet.relu
        self.maxpool = resnet.maxpool
        self.layer1 = resnet.layer1
        self.layer2 = resnet.layer2
        self.layer3 = resnet.layer3
        self.layer4 = resnet.layer4
        
    def forward(self, x):
        features = []
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        features.append(x)
        x = self.maxpool(x)
        x = self.layer1(x)
        features.append(x)
        x = self.layer2(x)
        features.append(x)
        x = self.layer3(x)
        features.append(x)
        x = self.layer4(x)
        features.append(x)
        return features

# ----------------------
# IMPROVED DATASET WITH NORMALIZED HISTOGRAMS
# ----------------------
class ImprovedDataset(Dataset):
    def __init__(self, dataframe, transform=None, image_size=256):
        self.dataframe = dataframe
        self.transform = transform
        self.image_size = image_size
        
        # Compute normalization stats
        all_hists = []
        for idx in range(min(500, len(dataframe))):
            row = dataframe.iloc[idx]
            hist = np.concatenate([row['r_hist'], row['g_hist'], row['b_hist']])
            all_hists.append(hist)
        
        all_hists = np.array(all_hists)
        self.hist_mean = np.mean(all_hists, axis=0)
        self.hist_std = np.std(all_hists, axis=0) + 1e-8
        
    def __len__(self):
        return len(self.dataframe)
    
    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]
        
        image = cv2.imread(row['image_path'])
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = cv2.resize(image, (self.image_size, self.image_size))
        
        if self.transform:
            image = self.transform(image)
        else:
            image = torch.FloatTensor(image).permute(2, 0, 1) / 255.0
        
        label = torch.LongTensor([row['label']])[0]
        
        # Normalize histogram features
        hist_features = np.concatenate([row['r_hist'], row['g_hist'], row['b_hist']])
        hist_features = (hist_features - self.hist_mean) / self.hist_std
        hist_features = np.clip(hist_features, -5, 5)
        hist_features = torch.FloatTensor(hist_features)
        
        return image, label, hist_features

# ----------------------
# IMPROVED HOVERNET-GNN MODEL
# ----------------------
HIST_SIZE = 33  # 11 bins x 3 channels

class ImprovedHoVerNetGNN(nn.Module):
    def __init__(self, num_classes=3, histogram_features=33):
        super(ImprovedHoVerNetGNN, self).__init__()
        
        self.encoder = HoVerNetEncoder(pretrained=True)
        
        self.cnn_proj = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3)
        )
        
        self.hist_proj = nn.Sequential(
            nn.Linear(histogram_features, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2)
        )
        
        self.classifier = nn.Sequential(
            nn.Linear(256 + 64, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.4),
            nn.Linear(128, 64),
            nn.ReLU(inplace=True),
            nn.Linear(64, num_classes)
        )
        
    def forward(self, images, hist_features):
        encoder_out = self.encoder(images)
        cnn_features = self.cnn_proj(encoder_out[-1])
        hist_out = self.hist_proj(hist_features)
        combined = torch.cat([cnn_features, hist_out], dim=1)
        return self.classifier(combined)

print("✅ Data loaded and models defined")

Loading data...


Loading folders:   0%|          | 0/40 [00:00<?, ?it/s]

Loaded 4000 samples
Train: 2800, Val: 600, Test: 600
✅ Data loaded and models defined


In [3]:
# ============================================================
# CELL 3: TRAIN AND SAVE FIXED HOVERNET-GNN
# ============================================================

print("="*60)
print("TRAINING IMPROVED HOVERNET-GNN MODEL")
print("="*60)

# Create datasets
print("\n[1/4] Creating datasets...")
train_dataset = ImprovedDataset(train_df, transform=train_transform, image_size=config.IMAGE_SIZE)
val_dataset = ImprovedDataset(val_df, transform=val_transform, image_size=config.IMAGE_SIZE)
test_dataset = ImprovedDataset(test_df, transform=val_transform, image_size=config.IMAGE_SIZE)

train_loader = DataLoader(train_dataset, batch_size=config.BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=config.BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=config.BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
print(f"  Train batches: {len(train_loader)}, Val batches: {len(val_loader)}, Test batches: {len(test_loader)}")

# Create model
print("\n[2/4] Training model...")
model = ImprovedHoVerNetGNN(num_classes=config.NUM_CLASSES, histogram_features=HIST_SIZE)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
encoder_params = list(model.encoder.parameters())
other_params = [p for n, p in model.named_parameters() if 'encoder' not in n]

optimizer = AdamW([
    {'params': encoder_params, 'lr': 5e-6},  # Very low LR for pretrained
    {'params': other_params, 'lr': 5e-5}
], weight_decay=1e-4)

scheduler = CosineAnnealingLR(optimizer, T_max=20, eta_min=1e-7)
scaler = GradScaler(enabled=config.USE_AMP)

# Training loop
best_val_acc = 0
best_model_state = None
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
patience, patience_counter = 7, 0

for epoch in range(20):
    # Training
    model.train()
    train_loss, correct, total = 0, 0, 0
    
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/20")
    for images, labels, hist_features in pbar:
        images = images.to(device)
        labels = labels.to(device)
        hist_features = hist_features.to(device)
        
        optimizer.zero_grad()
        
        with autocast(enabled=config.USE_AMP):
            outputs = model(images, hist_features)
            loss = criterion(outputs, labels)
        
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        
        train_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        
        pbar.set_postfix({'loss': f'{loss.item():.4f}', 'acc': f'{100.*correct/total:.1f}%'})
    
    train_loss /= len(train_loader)
    train_acc = 100. * correct / total
    
    # Validation
    model.eval()
    val_loss, correct, total = 0, 0, 0
    
    with torch.no_grad():
        for images, labels, hist_features in val_loader:
            images = images.to(device)
            labels = labels.to(device)
            hist_features = hist_features.to(device)
            
            outputs = model(images, hist_features)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
    
    val_loss /= len(val_loader)
    val_acc = 100. * correct / total
    
    scheduler.step()
    
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    
    print(f"  Epoch {epoch+1}: Train={train_acc:.2f}%, Val={val_acc:.2f}%")
    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_model_state = model.state_dict().copy()
        patience_counter = 0
        print(f"    ✓ New best! Val Acc: {val_acc:.2f}%")
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"    Early stopping at epoch {epoch+1}")
            break

# Load best model
if best_model_state:
    model.load_state_dict(best_model_state)

print(f"\n  Training complete! Best Val Accuracy: {best_val_acc:.2f}%")

# Evaluate on test set
print("\n[3/4] Evaluating on test set...")
model.eval()
all_preds, all_labels, all_probs = [], [], []

with torch.no_grad():
    for images, labels, hist_features in tqdm(test_loader, desc="Testing"):
        images = images.to(device)
        hist_features = hist_features.to(device)
        
        outputs = model(images, hist_features)
        probs = F.softmax(outputs, dim=1)
        _, predicted = outputs.max(1)
        
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.numpy())
        all_probs.extend(probs.cpu().numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)
all_probs = np.array(all_probs)

test_accuracy = accuracy_score(all_labels, all_preds)
test_f1 = f1_score(all_labels, all_preds, average='macro')
test_precision = precision_score(all_labels, all_preds, average='weighted')
test_recall = recall_score(all_labels, all_preds, average='weighted')

print(f"\n" + "="*60)
print("HOVERNET-GNN FIXED RESULTS")
print("="*60)
print(f"  Test Accuracy:  {test_accuracy*100:.2f}%")
print(f"  Test F1 Score:  {test_f1*100:.2f}%")
print(f"  Test Precision: {test_precision*100:.2f}%")
print(f"  Test Recall:    {test_recall*100:.2f}%")
print("="*60)

# Save model
print("\n[4/4] Saving model...")
torch.save({
    'model_state_dict': model.state_dict(),
    'history': history,
    'best_val_acc': best_val_acc,
    'test_accuracy': test_accuracy,
    'test_f1': test_f1,
    'histogram_size': HIST_SIZE
}, f"{config.OUTPUT_PATH}/models/hovernet_gnn_model_FIXED.pth")

# Save results
results = {
    'HoVerNet-GNN': {
        'accuracy': float(test_accuracy),
        'f1_macro': float(test_f1),
        'precision': float(test_precision),
        'recall': float(test_recall)
    }
}
with open(f"{config.OUTPUT_PATH}/results/hovernet_gnn_results.json", 'w') as f:
    json.dump(results, f, indent=2)

print(f"\n✅ Model saved to: {config.OUTPUT_PATH}/models/hovernet_gnn_model_FIXED.pth")
print(f"✅ Results saved to: {config.OUTPUT_PATH}/results/hovernet_gnn_results.json")

# Improvement summary
print(f"\n" + "="*60)
print("IMPROVEMENT SUMMARY")
print("="*60)
print(f"  Before Fix: 33.33% (random guessing)")
print(f"  After Fix:  {test_accuracy*100:.2f}%")
print(f"  Improvement: +{(test_accuracy - 0.3333)*100:.2f}%")
print("="*60)

# Zip for download
import zipfile
zip_path = f"{config.OUTPUT_PATH}/hovernet_gnn_fixed.zip"
with zipfile.ZipFile(zip_path, 'w') as zipf:
    zipf.write(f"{config.OUTPUT_PATH}/models/hovernet_gnn_model_FIXED.pth", "models/hovernet_gnn_model_FIXED.pth")
    zipf.write(f"{config.OUTPUT_PATH}/results/hovernet_gnn_results.json", "results/hovernet_gnn_results.json")

print(f"\n📦 Download: {zip_path}")
print("\nDone! Download the zip and replace in your Flask app.")

TRAINING IMPROVED HOVERNET-GNN MODEL

[1/4] Creating datasets...
  Train batches: 88, Val batches: 19, Test batches: 19

[2/4] Training model...
Downloading: "https://download.pytorch.org/models/resnet34-b627a593.pth" to /root/.cache/torch/hub/checkpoints/resnet34-b627a593.pth


100%|██████████| 83.3M/83.3M [00:00<00:00, 148MB/s] 


Epoch 1/20:   0%|          | 0/88 [00:00<?, ?it/s]

  Epoch 1: Train=44.75%, Val=70.17%
    ✓ New best! Val Acc: 70.17%


Epoch 2/20:   0%|          | 0/88 [00:00<?, ?it/s]

...

  Epoch 19: Train=88.32%, Val=89.33%


Epoch 20/20:   0%|          | 0/88 [00:00<?, ?it/s]

  Epoch 20: Train=87.71%, Val=88.50%

  Training complete! Best Val Accuracy: 91.67%

[3/4] Evaluating on test set...


Testing:   0%|          | 0/19 [00:00<?, ?it/s]


HOVERNET-GNN FIXED RESULTS
  Test Accuracy:  88.83%
  Test F1 Score:  88.67%
  Test Precision: 88.82%
  Test Recall:    88.83%

[4/4] Saving model...

✅ Model saved to: /kaggle/working//models/hovernet_gnn_model_FIXED.pth
✅ Results saved to: /kaggle/working//results/hovernet_gnn_results.json

IMPROVEMENT SUMMARY
  Before Fix: 33.33% (random guessing)
  After Fix:  88.83%
  Improvement: +55.50%

📦 Download: /kaggle/working//hovernet_gnn_fixed.zip

Done! Download the zip and replace in your Flask app.


In [5]:
# ============================================================
# CELL 3: TRAIN FULL HOVERNET-GNN (COMPLETE FIXED VERSION)
# ============================================================

print("="*60)
print("TRAINING FULL HOVERNET-GNN MODEL")
print("="*60)

# ============================================================
# FULL MODEL DEFINITION
# ============================================================

class FullHoVerNetGNN(nn.Module):
    """Full HoVerNet-GNN with encoder, decoder, segmentation, and hybrid classifier"""
    def __init__(self, num_classes=3, histogram_features=33):
        super(FullHoVerNetGNN, self).__init__()
        
        # Encoder
        self.encoder = HoVerNetEncoder(pretrained=True)
        
        # Decoder
        self.up4 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.conv4 = nn.Sequential(
            nn.Conv2d(512, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(inplace=True)
        )
        self.up3 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.conv3 = nn.Sequential(
            nn.Conv2d(256, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(inplace=True)
        )
        self.up2 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.conv2 = nn.Sequential(
            nn.Conv2d(128, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True)
        )
        self.up1 = nn.ConvTranspose2d(64, 64, kernel_size=2, stride=2)
        self.conv1 = nn.Sequential(
            nn.Conv2d(128, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True)
        )
        
        # Nucleus probability branch (with Sigmoid for output)
        self.np_branch = nn.Sequential(
            nn.Conv2d(64, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 1, kernel_size=1),
            nn.Sigmoid()
        )
        
        # HV gradient branch
        self.hv_branch = nn.Sequential(
            nn.Conv2d(64, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 2, kernel_size=1)
        )
        
        # CNN feature projection
        self.cnn_proj = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3)
        )
        
        # Histogram feature processor
        self.hist_proj = nn.Sequential(
            nn.Linear(histogram_features, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2)
        )
        
        # Combined classifier
        self.classifier = nn.Sequential(
            nn.Linear(256 + 64, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.4),
            nn.Linear(128, 64),
            nn.ReLU(inplace=True),
            nn.Linear(64, num_classes)
        )
    
    def decode(self, features):
        f0, f1, f2, f3, f4 = features
        x = self.up4(f4)
        x = torch.cat([x, f3], dim=1)
        x = self.conv4(x)
        x = self.up3(x)
        x = torch.cat([x, f2], dim=1)
        x = self.conv3(x)
        x = self.up2(x)
        x = torch.cat([x, f1], dim=1)
        x = self.conv2(x)
        x = self.up1(x)
        x = torch.cat([x, f0], dim=1)
        x = self.conv1(x)
        return x
        
    def forward(self, images, hist_features):
        # Encode
        encoder_features = self.encoder(images)
        
        # Decode for segmentation
        decoded = self.decode(encoder_features)
        decoded_up = F.interpolate(decoded, size=(images.size(2), images.size(3)), 
                                   mode='bilinear', align_corners=True)
        
        # Segmentation branches
        np_out = self.np_branch(decoded_up)
        hv_out = self.hv_branch(decoded_up)
        
        # Classification: CNN + Histogram hybrid
        cnn_features = self.cnn_proj(encoder_features[-1])
        hist_out = self.hist_proj(hist_features)
        combined = torch.cat([cnn_features, hist_out], dim=1)
        class_out = self.classifier(combined)
        
        return {
            'np': np_out,
            'hv': hv_out,
            'class': class_out
        }

# ============================================================
# CREATE DATASETS AND DATALOADERS
# ============================================================

print("\n[1/4] Creating datasets...")

train_dataset = ImprovedDataset(train_df, transform=train_transform, image_size=config.IMAGE_SIZE)
val_dataset = ImprovedDataset(val_df, transform=val_transform, image_size=config.IMAGE_SIZE)
test_dataset = ImprovedDataset(test_df, transform=val_transform, image_size=config.IMAGE_SIZE)

train_loader = DataLoader(train_dataset, batch_size=config.BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=config.BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=config.BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

print(f"  Train: {len(train_loader)} batches, Val: {len(val_loader)} batches, Test: {len(test_loader)} batches")

# ============================================================
# TRAINING WITH MULTI-TASK LOSS
# ============================================================

print("\n[2/4] Training with multi-task loss...")

model = FullHoVerNetGNN(num_classes=config.NUM_CLASSES, histogram_features=HIST_SIZE)
model = model.to(device)

# Separate learning rates
encoder_params = list(model.encoder.parameters())
other_params = [p for n, p in model.named_parameters() if 'encoder' not in n]

optimizer = AdamW([
    {'params': encoder_params, 'lr': 1e-5},
    {'params': other_params, 'lr': 1e-4}
], weight_decay=1e-4)

scheduler = CosineAnnealingLR(optimizer, T_max=25, eta_min=1e-7)
scaler = GradScaler(enabled=config.USE_AMP)

# Loss functions
classification_loss = nn.CrossEntropyLoss()
segmentation_loss = nn.MSELoss()  # FIXED: MSELoss works with autocast + Sigmoid
mse_loss = nn.MSELoss()

best_val_acc = 0
best_model_state = None
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
patience, patience_counter = 8, 0

for epoch in range(25):
    # ---- TRAINING ----
    model.train()
    train_loss_total, correct, total = 0, 0, 0
    
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/25")
    for images, labels, hist_features in pbar:
        images = images.to(device)
        labels = labels.to(device)
        hist_features = hist_features.to(device)
        
        optimizer.zero_grad()
        
        with autocast(enabled=config.USE_AMP):
            output = model(images, hist_features)
            
            # Classification loss (main objective)
            loss_cls = classification_loss(output['class'], labels)
            
            # Create pseudo ground truth for segmentation
            with torch.no_grad():
                gray = images.mean(dim=1, keepdim=True)
                pseudo_mask = (gray < gray.mean()).float()
            
            # Segmentation loss (auxiliary) - FIXED: using MSELoss
            loss_seg = segmentation_loss(output['np'], pseudo_mask)
            
            # Combined loss: prioritize classification
            loss = loss_cls + loss_seg * 0.1
        
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        
        train_loss_total += loss.item()
        _, predicted = output['class'].max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        
        pbar.set_postfix({
            'loss': f'{loss.item():.4f}', 
            'cls': f'{loss_cls.item():.4f}',
            'acc': f'{100.*correct/total:.1f}%'
        })
    
    train_loss = train_loss_total / len(train_loader)
    train_acc = 100. * correct / total
    
    # ---- VALIDATION ----
    model.eval()
    val_loss_total, correct, total = 0, 0, 0
    
    with torch.no_grad():
        for images, labels, hist_features in val_loader:
            images = images.to(device)
            labels = labels.to(device)
            hist_features = hist_features.to(device)
            
            output = model(images, hist_features)
            loss = classification_loss(output['class'], labels)
            
            val_loss_total += loss.item()
            _, predicted = output['class'].max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
    
    val_loss = val_loss_total / len(val_loader)
    val_acc = 100. * correct / total
    
    scheduler.step()
    
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    
    print(f"  Epoch {epoch+1}: Train={train_acc:.2f}%, Val={val_acc:.2f}%, Loss={val_loss:.4f}")
    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_model_state = model.state_dict().copy()
        patience_counter = 0
        print(f"    ✓ New best! Val Acc: {val_acc:.2f}%")
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"    Early stopping at epoch {epoch+1}")
            break

if best_model_state:
    model.load_state_dict(best_model_state)

print(f"\n  Best Val Accuracy: {best_val_acc:.2f}%")

# ============================================================
# EVALUATE ON TEST SET
# ============================================================

print("\n[3/4] Evaluating on test set...")

model.eval()
all_preds, all_labels, all_probs = [], [], []

with torch.no_grad():
    for images, labels, hist_features in tqdm(test_loader, desc="Testing"):
        images = images.to(device)
        hist_features = hist_features.to(device)
        
        output = model(images, hist_features)
        probs = F.softmax(output['class'], dim=1)
        _, predicted = output['class'].max(1)
        
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.numpy())
        all_probs.extend(probs.cpu().numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

test_accuracy = accuracy_score(all_labels, all_preds)
test_f1 = f1_score(all_labels, all_preds, average='macro')
test_precision = precision_score(all_labels, all_preds, average='weighted')
test_recall = recall_score(all_labels, all_preds, average='weighted')

print(f"\n{'='*60}")
print("HOVERNET-GNN RESULTS (FULL MODEL)")
print(f"{'='*60}")
print(f"  Test Accuracy:  {test_accuracy*100:.2f}%")
print(f"  Test F1 Score:  {test_f1*100:.2f}%")
print(f"  Test Precision: {test_precision*100:.2f}%")
print(f"  Test Recall:    {test_recall*100:.2f}%")
print(f"{'='*60}")

# ============================================================
# SAVE MODEL
# ============================================================

print("\n[4/4] Saving model...")

torch.save({
    'model_state_dict': model.state_dict(),
    'history': history,
    'best_val_acc': best_val_acc,
    'test_accuracy': test_accuracy,
    'histogram_size': HIST_SIZE
}, f"{config.OUTPUT_PATH}/models/hovernet_gnn_model_FIXED.pth")

results = {
    'HoVerNet-GNN': {
        'accuracy': float(test_accuracy),
        'f1_macro': float(test_f1),
        'precision': float(test_precision),
        'recall': float(test_recall)
    }
}
with open(f"{config.OUTPUT_PATH}/results/hovernet_gnn_results.json", 'w') as f:
    json.dump(results, f, indent=2)

# Zip for download
import zipfile
zip_path = f"{config.OUTPUT_PATH}/hovernet_gnn_fixed.zip"
with zipfile.ZipFile(zip_path, 'w') as zipf:
    zipf.write(f"{config.OUTPUT_PATH}/models/hovernet_gnn_model_FIXED.pth", "models/hovernet_gnn_model_FIXED.pth")
    zipf.write(f"{config.OUTPUT_PATH}/results/hovernet_gnn_results.json", "results/hovernet_gnn_results.json")

print(f"\n✅ Model saved: {config.OUTPUT_PATH}/models/hovernet_gnn_model_FIXED.pth")
print(f"✅ Results saved: {config.OUTPUT_PATH}/results/hovernet_gnn_results.json")
print(f"📦 Download: {zip_path}")

print(f"\n{'='*60}")
print(f"IMPROVEMENT: 33% → {test_accuracy*100:.2f}%")
print(f"{'='*60}")

TRAINING FULL HOVERNET-GNN MODEL

[1/4] Creating datasets...
  Train: 88 batches, Val: 19 batches, Test: 19 batches

[2/4] Training with multi-task loss...


Epoch 1/25:   0%|          | 0/88 [00:00<?, ?it/s]

  Epoch 1: Train=57.21%, Val=75.33%, Loss=0.7363
    ✓ New best! Val Acc: 75.33%


Epoch 2/25:   0%|          | 0/88 [00:00<?, ?it/s]

  Epoch 2: Train=75.64%, Val=81.33%, Loss=0.5042
    ✓ New best! Val Acc: 81.33%


...

  Epoch 22: Train=92.86%, Val=90.17%, Loss=0.2354


Epoch 23/25:   0%|          | 0/88 [00:00<?, ?it/s]

  Epoch 23: Train=92.75%, Val=90.50%, Loss=0.2259
    Early stopping at epoch 23

  Best Val Accuracy: 91.33%

[3/4] Evaluating on test set...


Testing:   0%|          | 0/19 [00:00<?, ?it/s]


HOVERNET-GNN RESULTS (FULL MODEL)
  Test Accuracy:  90.17%
  Test F1 Score:  90.04%
  Test Precision: 90.10%
  Test Recall:    90.17%

[4/4] Saving model...

✅ Model saved: /kaggle/working//models/hovernet_gnn_model_FIXED.pth
✅ Results saved: /kaggle/working//results/hovernet_gnn_results.json
📦 Download: /kaggle/working//hovernet_gnn_fixed.zip

IMPROVEMENT: 33% → 90.17%


In [ ]:
# ============================================================
# BREAST CANCER TUMOR SEGMENTATION AND ANALYSIS
# HoVerNet + GNN Hybrid Model
# ============================================================
# IMPORTANT: Run cells sequentially. Estimated GPU time: 3-4 hours
# ============================================================

# ============================================================
# SECTION 1: SETUP AND IMPORTS
# ============================================================

import os
import gc
import time
import random
import warnings
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict
import json

# PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam, AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR, ReduceLROnPlateau
from torch.cuda.amp import GradScaler, autocast

# Torchvision
import torchvision.transforms as transforms
from torchvision import models

# Sklearn
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    confusion_matrix, classification_report, roc_curve, auc,
    precision_recall_curve, accuracy_score, f1_score, precision_score, recall_score
)
from sklearn.preprocessing import LabelEncoder, StandardScaler

# OpenCV
import cv2

# Scipy for image processing
from scipy import ndimage
from scipy.ndimage import label as scipy_label

warnings.filterwarnings('ignore')

# Set seeds for reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

In [ ]:
# ============================================================
# SECTION 2: CONFIGURATION
# ============================================================

class Config:
    # Paths
    DATASET_PATH = '/kaggle/input/afssssssasf/'
    OUTPUT_PATH = '/kaggle/working/'

    # Data settings
    NUM_FOLDERS_TO_USE = 40  # Use 40 patient folders (balance speed vs accuracy)
    MAX_IMAGES_PER_FOLDER = 100  # Limit images per folder
    IMAGE_SIZE = 256  # Resize images for efficiency

    # Training settings
    BATCH_SIZE = 32
    NUM_EPOCHS = 15
    LEARNING_RATE = 1e-4
    WEIGHT_DECAY = 1e-5
    NUM_WORKERS = 2

    # Model settings
    NUM_CLASSES = 3  # Low, Medium, High cell density

    # Mixed precision
    USE_AMP = True

    # Checkpointing
    SAVE_BEST_ONLY = True

config = Config()

# Create output directories
os.makedirs(f"{config.OUTPUT_PATH}/models", exist_ok=True)
os.makedirs(f"{config.OUTPUT_PATH}/plots", exist_ok=True)
os.makedirs(f"{config.OUTPUT_PATH}/results", exist_ok=True)

print("Configuration loaded successfully!")

In [ ]:
# ============================================================
# SECTION 3: DATA LOADING AND PREPROCESSING
# ============================================================

print("\n" + "="*60)
print("LOADING AND PREPROCESSING DATA")
print("="*60)

def load_dataset(dataset_path, num_folders=40, max_images_per_folder=100):
    """Load images and CSV data from TCGA folders"""

    all_data = []

    # Get all TCGA folders
    all_folders = sorted([f for f in os.listdir(dataset_path)
                         if f.startswith('TCGA-') and os.path.isdir(os.path.join(dataset_path, f))])

    # Select subset of folders
    selected_folders = all_folders[:num_folders]
    print(f"Using {len(selected_folders)} out of {len(all_folders)} folders")

    for folder_name in tqdm(selected_folders, desc="Loading folders"):
        folder_path = os.path.join(dataset_path, folder_name)
        csv_path = os.path.join(folder_path, 'patch_info.csv')

        if not os.path.exists(csv_path):
            continue

        # Load CSV
        try:
            df = pd.read_csv(csv_path)
        except Exception as e:
            print(f"Error loading CSV for {folder_name}: {e}")
            continue

        # Get image files
        image_files = sorted([f for f in os.listdir(folder_path) if f.endswith('.jpg')])

        # Limit images per folder
        image_files = image_files[:max_images_per_folder]

        for img_file in image_files:
            img_id = int(os.path.splitext(img_file)[0])

            # Find matching row in CSV
            row = df[df['ID'] == float(img_id)]

            if len(row) == 0:
                continue

            row = row.iloc[0]

            all_data.append({
                'image_path': os.path.join(folder_path, img_file),
                'folder': folder_name,
                'image_id': img_id,
                'cell_index': row['cell index'],
                # Extract some histogram features for later use
                'r_hist': row[[f'R{i}' for i in range(0, 255, 25)]].values.astype(np.float32),
                'g_hist': row[[f'G{i}' for i in range(0, 255, 25)]].values.astype(np.float32),
                'b_hist': row[[f'B{i}' for i in range(0, 255, 25)]].values.astype(np.float32)
            })

    print(f"Total samples loaded: {len(all_data)}")
    return pd.DataFrame(all_data)

# Load data
df_data = load_dataset(config.DATASET_PATH, config.NUM_FOLDERS_TO_USE, config.MAX_IMAGES_PER_FOLDER)

In [ ]:
# ============================================================
# SECTION 4: CREATE LABELS FROM CELL INDEX
# ============================================================

print("\n" + "="*60)
print("CREATING LABELS")
print("="*60)

def create_labels(df, num_classes=3):
    """Create class labels based on cell index (cell density)"""

    cell_indices = df['cell_index'].values

    # Use percentile-based binning for balanced classes
    if num_classes == 3:
        # Low, Medium, High density
        percentiles = [0, 33, 66, 100]
        labels = ['Low Density', 'Medium Density', 'High Density']
    elif num_classes == 2:
        # Non-tumor vs Tumor (simplified)
        percentiles = [0, 50, 100]
        labels = ['Non-Tumor', 'Tumor']

    thresholds = np.percentile(cell_indices, percentiles[1:-1])
    print(f"Cell index thresholds: {thresholds}")

    # Assign labels
    df['label'] = 0
    for i, thresh in enumerate(thresholds):
        df.loc[df['cell_index'] >= thresh, 'label'] = i + 1

    df['label_name'] = df['label'].map({i: labels[i] for i in range(num_classes)})

    # Print distribution
    print("\nLabel distribution:")
    print(df['label_name'].value_counts())

    return df, labels

df_data, class_labels = create_labels(df_data, config.NUM_CLASSES)

# Cell index statistics
print("\nCell Index Statistics:")
print(df_data['cell_index'].describe())

# Visualize distribution
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.hist(df_data['cell_index'], bins=50, edgecolor='black', alpha=0.7)
plt.xlabel('Cell Index')
plt.ylabel('Count')
plt.title('Cell Index Distribution')

plt.subplot(1, 2, 2)
df_data['label_name'].value_counts().plot(kind='bar', color=['green', 'orange', 'red'], edgecolor='black')
plt.xlabel('Class')
plt.ylabel('Count')
plt.title('Class Distribution')
plt.xticks(rotation=45)

plt.tight_layout()
plt.savefig(f"{config.OUTPUT_PATH}/plots/data_distribution.png", dpi=150)
plt.show()


In [ ]:
# ============================================================
# SECTION 5: TRAIN/VAL/TEST SPLIT
# ============================================================

print("\n" + "="*60)
print("SPLITTING DATA")
print("="*60)

# Stratified split
train_df, temp_df = train_test_split(df_data, test_size=0.3, stratify=df_data['label'], random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df['label'], random_state=42)

print(f"Train size: {len(train_df)}")
print(f"Validation size: {len(val_df)}")
print(f"Test size: {len(test_df)}")

# Reset indices
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

In [ ]:
# ============================================================
# SECTION 6: DATASET CLASS
# ============================================================

class BreastCancerDataset(Dataset):
    """PyTorch Dataset for Breast Cancer Histopathology Images"""

    def __init__(self, dataframe, transform=None, image_size=256, return_histogram=False):
        self.dataframe = dataframe
        self.transform = transform
        self.image_size = image_size
        self.return_histogram = return_histogram

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]

        # Load image
        image = cv2.imread(row['image_path'])
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        # Resize
        image = cv2.resize(image, (self.image_size, self.image_size))

        # Apply transforms
        if self.transform:
            image = self.transform(image)
        else:
            image = torch.FloatTensor(image).permute(2, 0, 1) / 255.0

        label = torch.LongTensor([row['label']])[0]

        if self.return_histogram:
            # Combine histogram features
            hist_features = np.concatenate([row['r_hist'], row['g_hist'], row['b_hist']])
            hist_features = torch.FloatTensor(hist_features)
            return image, label, hist_features

        return image, label

# Define transforms
train_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Create datasets
train_dataset = BreastCancerDataset(train_df, transform=train_transform,
                                     image_size=config.IMAGE_SIZE, return_histogram=True)
val_dataset = BreastCancerDataset(val_df, transform=val_transform,
                                   image_size=config.IMAGE_SIZE, return_histogram=True)
test_dataset = BreastCancerDataset(test_df, transform=val_transform,
                                    image_size=config.IMAGE_SIZE, return_histogram=True)

# Create dataloaders
train_loader = DataLoader(train_dataset, batch_size=config.BATCH_SIZE,
                          shuffle=True, num_workers=config.NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=config.BATCH_SIZE,
                        shuffle=False, num_workers=config.NUM_WORKERS, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=config.BATCH_SIZE,
                         shuffle=False, num_workers=config.NUM_WORKERS, pin_memory=True)

print(f"DataLoaders created successfully!")
print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}, Test batches: {len(test_loader)}")

In [ ]:
!pip install torch-geometric \
  torch-scatter \
  torch-sparse \
  torch-cluster \
  torch-spline-conv \
  -f https://data.pyg.org/whl/torch-2.6.0+cu121.html


In [ ]:
import torch
print(torch.__version__)
print(torch.version.cuda)


In [ ]:
# ============================================================
# SECTION 7: MODEL DEFINITIONS
# ============================================================

print("\n" + "="*60)
print("DEFINING MODELS")
print("="*60)

# 7.1 Simple CNN Model
class SimpleCNN(nn.Module):
    """Simple CNN for baseline comparison"""

    def __init__(self, num_classes=3):
        super(SimpleCNN, self).__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((1, 1))
        )

        self.classifier = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        x = self.classifier(x)
        return x

# 7.2 ResNet Model
class ResNetClassifier(nn.Module):
    """ResNet-based classifier with pretrained weights"""

    def __init__(self, num_classes=3, pretrained=True):
        super(ResNetClassifier, self).__init__()

        # Load pretrained ResNet50
        self.resnet = models.resnet50(weights='IMAGENET1K_V1' if pretrained else None)

        # Modify final layer
        in_features = self.resnet.fc.in_features
        self.resnet.fc = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(in_features, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        return self.resnet(x)

    def get_features(self, x):
        """Get features before final layer (for GNN)"""
        x = self.resnet.conv1(x)
        x = self.resnet.bn1(x)
        x = self.resnet.relu(x)
        x = self.resnet.maxpool(x)
        x = self.resnet.layer1(x)
        x = self.resnet.layer2(x)
        x = self.resnet.layer3(x)
        x = self.resnet.layer4(x)
        x = self.resnet.avgpool(x)
        x = torch.flatten(x, 1)
        return x

# 7.3 HoVerNet-Style Encoder-Decoder for Nucleus Segmentation
class ConvBlock(nn.Module):
    """Convolutional block with BatchNorm and ReLU"""

    def __init__(self, in_channels, out_channels):
        super(ConvBlock, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.conv(x)

class HoVerNetEncoder(nn.Module):
    """Encoder based on ResNet (similar to HoVerNet)"""

    def __init__(self, pretrained=True):
        super(HoVerNetEncoder, self).__init__()

        resnet = models.resnet34(weights='IMAGENET1K_V1' if pretrained else None)

        self.conv1 = resnet.conv1
        self.bn1 = resnet.bn1
        self.relu = resnet.relu
        self.maxpool = resnet.maxpool

        self.layer1 = resnet.layer1  # 64 channels
        self.layer2 = resnet.layer2  # 128 channels
        self.layer3 = resnet.layer3  # 256 channels
        self.layer4 = resnet.layer4  # 512 channels

    def forward(self, x):
        features = []

        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        features.append(x)  # 64 channels

        x = self.maxpool(x)
        x = self.layer1(x)
        features.append(x)  # 64 channels

        x = self.layer2(x)
        features.append(x)  # 128 channels

        x = self.layer3(x)
        features.append(x)  # 256 channels

        x = self.layer4(x)
        features.append(x)  # 512 channels

        return features

class HoVerNetDecoder(nn.Module):
    """Decoder for HoVerNet-style segmentation"""

    def __init__(self, encoder_channels=[64, 64, 128, 256, 512]):
        super(HoVerNetDecoder, self).__init__()

        self.up4 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.conv4 = ConvBlock(512, 256)

        self.up3 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.conv3 = ConvBlock(256, 128)

        self.up2 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.conv2 = ConvBlock(128, 64)

        self.up1 = nn.ConvTranspose2d(64, 64, kernel_size=2, stride=2)
        self.conv1 = ConvBlock(128, 64)

    def forward(self, features):
        # features = [f0, f1, f2, f3, f4] from encoder
        f0, f1, f2, f3, f4 = features

        x = self.up4(f4)
        x = torch.cat([x, f3], dim=1)
        x = self.conv4(x)

        x = self.up3(x)
        x = torch.cat([x, f2], dim=1)
        x = self.conv3(x)

        x = self.up2(x)
        x = torch.cat([x, f1], dim=1)
        x = self.conv2(x)

        x = self.up1(x)
        x = torch.cat([x, f0], dim=1)
        x = self.conv1(x)

        return x

class HoVerNet(nn.Module):
    """
    HoVerNet-style model for nucleus segmentation and classification.
    Outputs:
    - Nucleus probability map
    - Horizontal gradient map
    - Vertical gradient map
    - Classification output
    """

    def __init__(self, num_classes=3, pretrained=True):
        super(HoVerNet, self).__init__()

        self.encoder = HoVerNetEncoder(pretrained=pretrained)
        self.decoder = HoVerNetDecoder()

        # Nucleus probability branch
        self.np_branch = nn.Sequential(
            nn.Conv2d(64, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 1, kernel_size=1),
            nn.Sigmoid()
        )

        # Horizontal-Vertical gradient branch
        self.hv_branch = nn.Sequential(
            nn.Conv2d(64, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 2, kernel_size=1)  # 2 channels: H and V
        )

        # Classification branch (for patch-level classification)
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(64, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        # Encode
        features = self.encoder(x)

        # Decode
        decoded = self.decoder(features)

        # Upsample to original size
        decoded_up = F.interpolate(decoded, size=(x.size(2), x.size(3)),
                                   mode='bilinear', align_corners=True)

        # Output branches
        np_out = self.np_branch(decoded_up)  # Nucleus probability
        hv_out = self.hv_branch(decoded_up)  # H-V gradients
        class_out = self.classifier(decoded)  # Classification

        return {
            'np': np_out,        # [B, 1, H, W] - Nucleus probability
            'hv': hv_out,        # [B, 2, H, W] - Horizontal-Vertical gradients
            'class': class_out  # [B, num_classes] - Classification logits
        }

    def get_segmentation_features(self, x):
        """Get features for GNN"""
        features = self.encoder(x)
        decoded = self.decoder(features)
        return decoded

# 7.4 Graph Neural Network Components
try:
    from torch_geometric.nn import GCNConv, SAGEConv, GATConv, global_mean_pool
    from torch_geometric.data import Data, Batch
    TORCH_GEOMETRIC_AVAILABLE = True
    print("PyTorch Geometric is available!")
except ImportError:
    TORCH_GEOMETRIC_AVAILABLE = False
    print("PyTorch Geometric not available. Installing...")
    os.system('pip install torch-geometric torch-scatter torch-sparse -q')
    try:
        from torch_geometric.nn import GCNConv, SAGEConv, GATConv, global_mean_pool
        from torch_geometric.data import Data, Batch
        TORCH_GEOMETRIC_AVAILABLE = True
        print("PyTorch Geometric installed successfully!")
    except:
        print("Failed to install PyTorch Geometric. GNN will use fallback.")

if TORCH_GEOMETRIC_AVAILABLE:
    class GraphSAGEClassifier(nn.Module):
        """GraphSAGE model for spatial cell relationship analysis"""

        def __init__(self, in_features=64, hidden_features=128, num_classes=3):
            super(GraphSAGEClassifier, self).__init__()

            self.conv1 = SAGEConv(in_features, hidden_features)
            self.conv2 = SAGEConv(hidden_features, hidden_features)
            self.conv3 = SAGEConv(hidden_features, hidden_features // 2)

            self.classifier = nn.Sequential(
                nn.Linear(hidden_features // 2, 64),
                nn.ReLU(inplace=True),
                nn.Dropout(0.5),
                nn.Linear(64, num_classes)
            )

        def forward(self, x, edge_index, batch):
            x = F.relu(self.conv1(x, edge_index))
            x = F.dropout(x, p=0.3, training=self.training)
            x = F.relu(self.conv2(x, edge_index))
            x = F.dropout(x, p=0.3, training=self.training)
            x = F.relu(self.conv3(x, edge_index))

            # Global mean pooling
            x = global_mean_pool(x, batch)

            # Classify
            x = self.classifier(x)
            return x

    class GATClassifier(nn.Module):
        """Graph Attention Network for spatial cell analysis"""

        def __init__(self, in_features=64, hidden_features=128, num_classes=3, heads=4):
            super(GATClassifier, self).__init__()

            self.conv1 = GATConv(in_features, hidden_features // heads, heads=heads)
            self.conv2 = GATConv(hidden_features, hidden_features // heads, heads=heads)
            self.conv3 = GATConv(hidden_features, hidden_features // 2, heads=1)

            self.classifier = nn.Sequential(
                nn.Linear(hidden_features // 2, 64),
                nn.ReLU(inplace=True),
                nn.Dropout(0.5),
                nn.Linear(64, num_classes)
            )

        def forward(self, x, edge_index, batch):
            x = F.relu(self.conv1(x, edge_index))
            x = F.dropout(x, p=0.3, training=self.training)
            x = F.relu(self.conv2(x, edge_index))
            x = F.dropout(x, p=0.3, training=self.training)
            x = F.relu(self.conv3(x, edge_index))

            x = global_mean_pool(x, batch)
            x = self.classifier(x)
            return x

# 7.5 Hybrid HoVerNet + GNN Model
class HybridHoVerNetGNN(nn.Module):
    """Hybrid model combining HoVerNet features with GNN for spatial analysis"""

    def __init__(self, num_classes=3, histogram_features=30):
        super(HybridHoVerNetGNN, self).__init__()

        # CNN feature extractor (from HoVerNet encoder)
        self.encoder = HoVerNetEncoder(pretrained=True)

        # Feature projection
        self.feature_proj = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(512, 256),
            nn.ReLU(inplace=True)
        )

        # Histogram feature processor
        self.hist_processor = nn.Sequential(
            nn.Linear(histogram_features, 64),
            nn.ReLU(inplace=True),
            nn.Linear(64, 64),
            nn.ReLU(inplace=True)
        )

        # Combined classifier
        self.classifier = nn.Sequential(
            nn.Linear(256 + 64, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(128, num_classes)
        )

    def forward(self, images, hist_features):
        # Extract image features
        features = self.encoder(images)
        img_features = self.feature_proj(features[-1])

        # Process histogram features
        hist_out = self.hist_processor(hist_features)

        # Combine and classify
        combined = torch.cat([img_features, hist_out], dim=1)
        output = self.classifier(combined)

        return output

print("All models defined successfully!")

In [ ]:
# ============================================================
# SECTION 7.6: RNN/LSTM MODEL (For Comparison)
# ============================================================

class HistogramRNN(nn.Module):
    """
    RNN/LSTM model that processes color histogram sequences.
    Treats R, G, B histograms as a sequence for comparison.
    """

    def __init__(self, input_size=255, hidden_size=128, num_layers=2, num_classes=3):
        super(HistogramRNN, self).__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        # CNN feature extractor for image
        self.cnn = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(4, 4),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(4, 4),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((4, 4))
        )

        # LSTM for sequential processing
        self.lstm = nn.LSTM(
            input_size=128 * 16,  # Flattened CNN features
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=0.3,
            bidirectional=True
        )

        # Classifier
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size * 2, 128),  # *2 for bidirectional
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        batch_size = x.size(0)

        # Extract CNN features
        features = self.cnn(x)
        features = features.view(batch_size, 1, -1)  # [B, 1, 128*16]

        # Repeat for sequence (simulate temporal/spatial sequence)
        features = features.repeat(1, 4, 1)  # [B, 4, 128*16]

        # LSTM
        lstm_out, (h_n, c_n) = self.lstm(features)

        # Use last hidden state
        out = torch.cat([h_n[-2], h_n[-1]], dim=1)  # Bidirectional

        # Classify
        out = self.classifier(out)
        return out


# ============================================================
# SECTION 7.7: COMPLETE GNN WITH GRAPH CONSTRUCTION
# ============================================================

class NucleiGraphDataset(Dataset):
    """Dataset that builds graphs from detected nuclei"""

    def __init__(self, dataframe, transform=None, image_size=256):
        self.dataframe = dataframe
        self.transform = transform
        self.image_size = image_size

    def __len__(self):
        return len(self.dataframe)

    def detect_nuclei_from_image(self, image):
        """Detect nuclei using color-based segmentation"""
        # Convert to grayscale
        gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)

        # Apply adaptive thresholding
        binary = cv2.adaptiveThreshold(
            gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
            cv2.THRESH_BINARY_INV, 21, 5
        )

        # Morphological operations
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
        binary = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel, iterations=2)
        binary = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel, iterations=2)

        # Find contours
        contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        nuclei = []
        for contour in contours:
            area = cv2.contourArea(contour)
            if 50 < area < 5000:  # Filter by area
                M = cv2.moments(contour)
                if M["m00"] > 0:
                    cx = int(M["m10"] / M["m00"])
                    cy = int(M["m01"] / M["m00"])

                    # Extract local features
                    x, y, w, h = cv2.boundingRect(contour)
                    roi = image[max(0,y-5):min(image.shape[0],y+h+5),
                               max(0,x-5):min(image.shape[1],x+w+5)]

                    if roi.size > 0:
                        mean_color = roi.mean(axis=(0,1)) / 255.0
                    else:
                        mean_color = np.array([0.5, 0.5, 0.5])

                    nuclei.append({
                        'centroid': (cx, cy),
                        'area': area,
                        'perimeter': cv2.arcLength(contour, True),
                        'mean_color': mean_color
                    })

        return nuclei

    def build_graph(self, nuclei, k_neighbors=6, max_distance=80):
        """Build graph from nuclei detections"""
        if len(nuclei) < 3:
            # Return dummy graph
            return {
                'node_features': np.zeros((1, 8), dtype=np.float32),
                'edge_index': np.array([[0], [0]], dtype=np.int64),
                'num_nodes': 1
            }

        # Extract centroids
        centroids = np.array([n['centroid'] for n in nuclei])
        areas = np.array([n['area'] for n in nuclei])
        perimeters = np.array([n['perimeter'] for n in nuclei])
        colors = np.array([n['mean_color'] for n in nuclei])

        # Normalize features
        centroids_norm = centroids / self.image_size
        areas_norm = (areas - areas.mean()) / (areas.std() + 1e-6)
        perimeters_norm = (perimeters - perimeters.mean()) / (perimeters.std() + 1e-6)

        # Node features: [x, y, area, perimeter, R, G, B, circularity]
        circularity = 4 * np.pi * areas / (perimeters ** 2 + 1e-6)

        node_features = np.column_stack([
            centroids_norm,
            areas_norm.reshape(-1, 1),
            perimeters_norm.reshape(-1, 1),
            colors,
            circularity.reshape(-1, 1)
        ]).astype(np.float32)

        # Build edges using k-NN
        from scipy.spatial.distance import cdist
        distances = cdist(centroids, centroids)

        edges = []
        for i in range(len(nuclei)):
            sorted_indices = np.argsort(distances[i])
            for j in sorted_indices[1:k_neighbors+1]:
                if distances[i, j] < max_distance:
                    edges.append([i, j])

        if len(edges) == 0:
            # Add self-loops if no edges
            edges = [[i, i] for i in range(len(nuclei))]

        edge_index = np.array(edges, dtype=np.int64).T

        return {
            'node_features': node_features,
            'edge_index': edge_index,
            'num_nodes': len(nuclei)
        }

    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]

        # Load image
        image = cv2.imread(row['image_path'])
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = cv2.resize(image, (self.image_size, self.image_size))

        # Detect nuclei
        nuclei = self.detect_nuclei_from_image(image)

        # Build graph
        graph_data = self.build_graph(nuclei)

        # Transform image
        if self.transform:
            image_tensor = self.transform(image)
        else:
            image_tensor = torch.FloatTensor(image).permute(2, 0, 1) / 255.0

        label = torch.LongTensor([row['label']])[0]

        return {
            'image': image_tensor,
            'label': label,
            'node_features': torch.FloatTensor(graph_data['node_features']),
            'edge_index': torch.LongTensor(graph_data['edge_index']),
            'num_nodes': graph_data['num_nodes'],
            'num_nuclei': len(nuclei)
        }


# GNN Classifier (Updated to work with our graph format)
if TORCH_GEOMETRIC_AVAILABLE:

    class CompleteGraphSAGE(nn.Module):
        """Complete GraphSAGE model for nuclei graph classification"""

        def __init__(self, in_features=8, hidden_features=64, num_classes=3):
            super(CompleteGraphSAGE, self).__init__()

            self.conv1 = SAGEConv(in_features, hidden_features)
            self.bn1 = nn.BatchNorm1d(hidden_features)

            self.conv2 = SAGEConv(hidden_features, hidden_features)
            self.bn2 = nn.BatchNorm1d(hidden_features)

            self.conv3 = SAGEConv(hidden_features, hidden_features)
            self.bn3 = nn.BatchNorm1d(hidden_features)

            self.classifier = nn.Sequential(
                nn.Linear(hidden_features, 64),
                nn.ReLU(inplace=True),
                nn.Dropout(0.5),
                nn.Linear(64, num_classes)
            )

        def forward(self, x, edge_index, batch):
            # Layer 1
            x = self.conv1(x, edge_index)
            x = self.bn1(x)
            x = F.relu(x)
            x = F.dropout(x, p=0.3, training=self.training)

            # Layer 2
            x = self.conv2(x, edge_index)
            x = self.bn2(x)
            x = F.relu(x)
            x = F.dropout(x, p=0.3, training=self.training)

            # Layer 3
            x = self.conv3(x, edge_index)
            x = self.bn3(x)
            x = F.relu(x)

            # Global pooling
            x = global_mean_pool(x, batch)

            # Classify
            out = self.classifier(x)
            return out


    class HybridHoVerNetGraphSAGE(nn.Module):
        """
        Complete Hybrid Model: HoVerNet encoder + GraphSAGE
        This is the main model that combines both approaches.
        """

        def __init__(self, num_classes=3, gnn_features=8, gnn_hidden=64):
            super(HybridHoVerNetGraphSAGE, self).__init__()

            # HoVerNet Encoder
            self.encoder = HoVerNetEncoder(pretrained=True)

            # CNN feature projection
            self.cnn_proj = nn.Sequential(
                nn.AdaptiveAvgPool2d((1, 1)),
                nn.Flatten(),
                nn.Linear(512, 128),
                nn.ReLU(inplace=True)
            )

            # Graph Neural Network
            self.gnn_conv1 = SAGEConv(gnn_features, gnn_hidden)
            self.gnn_conv2 = SAGEConv(gnn_hidden, gnn_hidden)
            self.gnn_proj = nn.Linear(gnn_hidden, 64)

            # Combined classifier
            self.classifier = nn.Sequential(
                nn.Linear(128 + 64, 128),
                nn.ReLU(inplace=True),
                nn.Dropout(0.5),
                nn.Linear(128, 64),
                nn.ReLU(inplace=True),
                nn.Linear(64, num_classes)
            )

        def forward(self, images, node_features, edge_index, batch):
            # CNN features from HoVerNet encoder
            encoder_features = self.encoder(images)
            cnn_features = self.cnn_proj(encoder_features[-1])

            # GNN features
            x = F.relu(self.gnn_conv1(node_features, edge_index))
            x = F.dropout(x, p=0.3, training=self.training)
            x = F.relu(self.gnn_conv2(x, edge_index))
            x = global_mean_pool(x, batch)
            gnn_features = self.gnn_proj(x)

            # Combine and classify
            combined = torch.cat([cnn_features, gnn_features], dim=1)
            output = self.classifier(combined)

            return output


# ============================================================
# SECTION 7.8: CELL DENSITY AND SPATIAL RELATIONSHIP VISUALIZATION
# ============================================================

def create_spatial_relationship_map(image_path, nuclei, edges, output_size=256):
    """
    Create visualization of spatial relationships between cells.
    Shows nuclei as nodes and edges as connections.
    """
    # Load image
    image = cv2.imread(image_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image = cv2.resize(image, (output_size, output_size))

    # Create overlay
    overlay = image.copy()

    # Draw edges
    if len(edges) > 0:
        for i in range(edges.shape[1]):
            src, dst = edges[:, i]
            if src < len(nuclei) and dst < len(nuclei):
                pt1 = tuple(map(int, nuclei[src]['centroid']))
                pt2 = tuple(map(int, nuclei[dst]['centroid']))
                cv2.line(overlay, pt1, pt2, (0, 255, 255), 1, cv2.LINE_AA)

    # Draw nuclei
    for i, nucleus in enumerate(nuclei):
        center = tuple(map(int, nucleus['centroid']))
        radius = int(np.sqrt(nucleus['area'] / np.pi))
        cv2.circle(overlay, center, radius, (255, 0, 0), 2)
        cv2.circle(overlay, center, 3, (0, 255, 0), -1)

    # Blend
    result = cv2.addWeighted(image, 0.6, overlay, 0.4, 0)

    return result


def create_cell_density_heatmap(image_shape, nuclei, kernel_size=51):
    """
    Create cell density heatmap based on nuclei positions.
    """
    # Create empty density map
    density_map = np.zeros(image_shape[:2], dtype=np.float32)

    # Place points at nuclei locations
    for nucleus in nuclei:
        x, y = map(int, nucleus['centroid'])
        if 0 <= x < image_shape[1] and 0 <= y < image_shape[0]:
            density_map[y, x] = 1

    # Apply Gaussian blur to create density
    density_map = cv2.GaussianBlur(density_map, (kernel_size, kernel_size), 0)

    # Normalize
    if density_map.max() > 0:
        density_map = density_map / density_map.max()

    # Apply colormap
    density_colored = cv2.applyColorMap(
        (density_map * 255).astype(np.uint8),
        cv2.COLORMAP_JET
    )
    density_colored = cv2.cvtColor(density_colored, cv2.COLOR_BGR2RGB)

    return density_map, density_colored


In [ ]:
# ============================================================
# SECTION 8: NUCLEUS DETECTION AND GRAPH CONSTRUCTION
# ============================================================

def detect_nuclei(image, min_size=50, max_size=5000):
    """
    Detect nuclei in histopathology image using color-based segmentation.
    Returns centroids and properties of detected nuclei.
    """
    # Convert to different color spaces
    if len(image.shape) == 2:
        gray = image
    else:
        # Convert to grayscale and extract different channels
        gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
        hsv = cv2.cvtColor(image, cv2.COLOR_RGB2HSV)

        # Use hematoxylin-like channel (blue-ish nuclei)
        # Nuclei typically have high blue, low red
        blue_ratio = image[:,:,2].astype(float) / (image[:,:,0].astype(float) + 1)

    # Threshold
    _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

    # Morphological operations
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    binary = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel, iterations=2)
    binary = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel, iterations=2)

    # Connected components
    labeled_array, num_features = scipy_label(binary)

    nuclei = []
    for i in range(1, num_features + 1):
        # Get region
        region = (labeled_array == i)
        area = np.sum(region)

        if min_size < area < max_size:
            # Get centroid
            coords = np.where(region)
            centroid_y = np.mean(coords[0])
            centroid_x = np.mean(coords[1])

            # Get bounding box
            min_y, max_y = np.min(coords[0]), np.max(coords[0])
            min_x, max_x = np.min(coords[1]), np.max(coords[1])

            nuclei.append({
                'centroid': (centroid_x, centroid_y),
                'area': area,
                'bbox': (min_x, min_y, max_x, max_y)
            })

    return nuclei

def build_graph_from_nuclei(nuclei, image_features=None, k_neighbors=5, max_distance=100):
    """
    Build a graph from detected nuclei.
    Nodes: nuclei
    Edges: k-nearest neighbors within max_distance
    """
    if len(nuclei) < 2:
        return None

    # Get centroids
    centroids = np.array([n['centroid'] for n in nuclei])
    areas = np.array([n['area'] for n in nuclei])

    # Normalize features
    areas_norm = (areas - areas.mean()) / (areas.std() + 1e-6)

    # Create node features: [x, y, area, ...]
    node_features = np.column_stack([
        centroids[:, 0] / 256,  # Normalized x
        centroids[:, 1] / 256,  # Normalized y
        areas_norm
    ])

    if image_features is not None:
        node_features = np.column_stack([node_features,
                                         np.tile(image_features, (len(nuclei), 1))])

    # Build edges using k-nearest neighbors
    from scipy.spatial.distance import cdist
    distances = cdist(centroids, centroids)

    edges = []
    for i in range(len(nuclei)):
        # Get k nearest neighbors
        sorted_indices = np.argsort(distances[i])
        for j in sorted_indices[1:k_neighbors+1]:  # Skip self
            if distances[i, j] < max_distance:
                edges.append([i, j])

    if len(edges) == 0:
        return None

    edge_index = np.array(edges).T

    return {
        'node_features': node_features,
        'edge_index': edge_index,
        'num_nodes': len(nuclei)
    }

print("Nucleus detection and graph construction functions defined!")

In [ ]:
# ============================================================
# SECTION 9: TRAINING UTILITIES
# ============================================================

class EarlyStopping:
    """Early stopping to prevent overfitting"""

    def __init__(self, patience=5, min_delta=0.001, mode='min'):
        self.patience = patience
        self.min_delta = min_delta
        self.mode = mode
        self.counter = 0
        self.best_score = None
        self.early_stop = False

    def __call__(self, score):
        if self.best_score is None:
            self.best_score = score
            return False

        if self.mode == 'min':
            improved = score < self.best_score - self.min_delta
        else:
            improved = score > self.best_score + self.min_delta

        if improved:
            self.best_score = score
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
                return True
        return False

def train_epoch(model, dataloader, criterion, optimizer, scaler, device, use_histogram=False):
    """Train for one epoch"""
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    pbar = tqdm(dataloader, desc="Training", leave=False)

    for batch in pbar:
        if use_histogram:
            images, labels, hist_features = batch
            images = images.to(device)
            labels = labels.to(device)
            hist_features = hist_features.to(device)
        else:
            images, labels = batch[:2]
            images = images.to(device)
            labels = labels.to(device)
            hist_features = None

        optimizer.zero_grad()

        with autocast(enabled=config.USE_AMP):
            if use_histogram and hasattr(model, 'hist_processor'):
                outputs = model(images, hist_features)
            else:
                outputs = model(images)
                if isinstance(outputs, dict):
                    outputs = outputs['class']

            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

        pbar.set_postfix({'loss': f'{loss.item():.4f}', 'acc': f'{100.*correct/total:.2f}%'})

    epoch_loss = running_loss / len(dataloader)
    epoch_acc = 100. * correct / total

    return epoch_loss, epoch_acc

def validate_epoch(model, dataloader, criterion, device, use_histogram=False):
    """Validate for one epoch"""
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []
    all_probs = []

    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Validating", leave=False):
            if use_histogram:
                images, labels, hist_features = batch
                images = images.to(device)
                labels = labels.to(device)
                hist_features = hist_features.to(device)
            else:
                images, labels = batch[:2]
                images = images.to(device)
                labels = labels.to(device)
                hist_features = None

            with autocast(enabled=config.USE_AMP):
                if use_histogram and hasattr(model, 'hist_processor'):
                    outputs = model(images, hist_features)
                else:
                    outputs = model(images)
                    if isinstance(outputs, dict):
                        outputs = outputs['class']

                loss = criterion(outputs, labels)

            running_loss += loss.item()
            probs = F.softmax(outputs, dim=1)
            _, predicted = outputs.max(1)

            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    epoch_loss = running_loss / len(dataloader)
    epoch_acc = 100. * correct / total

    return epoch_loss, epoch_acc, np.array(all_preds), np.array(all_labels), np.array(all_probs)

def train_model(model, train_loader, val_loader, model_name, num_epochs=15,
                learning_rate=1e-4, use_histogram=False):
    """Complete training loop for a model"""

    print(f"\n{'='*60}")
    print(f"Training: {model_name}")
    print(f"{'='*60}")

    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = AdamW(model.parameters(), lr=learning_rate, weight_decay=config.WEIGHT_DECAY)
    scheduler = CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=1e-6)
    scaler = GradScaler(enabled=config.USE_AMP)
    early_stopping = EarlyStopping(patience=5, mode='max')

    history = {
        'train_loss': [], 'train_acc': [],
        'val_loss': [], 'val_acc': []
    }

    best_val_acc = 0
    best_model_state = None

    start_time = time.time()

    for epoch in range(num_epochs):
        print(f"\nEpoch {epoch+1}/{num_epochs}")
        print("-" * 30)

        # Train
        train_loss, train_acc = train_epoch(
            model, train_loader, criterion, optimizer, scaler, device, use_histogram
        )

        # Validate
        val_loss, val_acc, _, _, _ = validate_epoch(
            model, val_loader, criterion, device, use_histogram
        )

        # Update scheduler
        scheduler.step()

        # Log
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)

        print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%")
        print(f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%")

        # Save best model
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_model_state = model.state_dict().copy()
            print(f"✓ New best model! Val Acc: {val_acc:.2f}%")

        # Early stopping
        if early_stopping(val_acc):
            print(f"Early stopping triggered at epoch {epoch+1}")
            break

    training_time = time.time() - start_time
    print(f"\nTraining completed in {training_time/60:.2f} minutes")
    print(f"Best Validation Accuracy: {best_val_acc:.2f}%")

    # Load best model
    if best_model_state is not None:
        model.load_state_dict(best_model_state)

    return model, history, best_val_acc

print("Training utilities defined!")


In [ ]:
# ============================================================
# SECTION 10: TRAIN ALL MODELS (FIXED VERSION)
# ============================================================

print("\n" + "="*60)
print("FIXING CONFIGURATION AND TRAINING ALL MODELS")
print("="*60)

# ============================================================
# FIX 1: Recreate DataLoaders with num_workers=0
# This fixes the multiprocessing assertion errors
# ============================================================

print("\n[FIX] Recreating DataLoaders with num_workers=0...")

train_loader = DataLoader(
    train_dataset,
    batch_size=config.BATCH_SIZE,
    shuffle=True,
    num_workers=0,  # FIXED: Set to 0 to avoid multiprocessing issues
    pin_memory=True
)
val_loader = DataLoader(
    val_dataset,
    batch_size=config.BATCH_SIZE,
    shuffle=False,
    num_workers=0,  # FIXED
    pin_memory=True
)
test_loader = DataLoader(
    test_dataset,
    batch_size=config.BATCH_SIZE,
    shuffle=False,
    num_workers=0,  # FIXED
    pin_memory=True
)

print("✓ DataLoaders recreated with num_workers=0")

# ============================================================
# FIX 2: Check actual histogram feature size
# ============================================================

print("\n[FIX] Checking histogram feature size...")

sample = train_dataset[0]
if len(sample) == 3:
    _, _, hist_sample = sample
    ACTUAL_HIST_SIZE = hist_sample.shape[0]
else:
    ACTUAL_HIST_SIZE = 33  # Default: 11 bins * 3 channels

print(f"✓ Actual histogram feature size: {ACTUAL_HIST_SIZE}")

# ============================================================
# FIX 3: Redefine HybridHoVerNetGNN with correct histogram size
# ============================================================

print("\n[FIX] Redefining HybridHoVerNetGNN with correct histogram size...")

class HybridHoVerNetGNN(nn.Module):
    """
    Fixed Hybrid model combining HoVerNet features with histogram features.
    Histogram size is dynamically set based on actual data.
    """

    def __init__(self, num_classes=3, histogram_features=33):  # FIXED: Default to 33
        super(HybridHoVerNetGNN, self).__init__()

        # CNN feature extractor (from HoVerNet encoder)
        self.encoder = HoVerNetEncoder(pretrained=True)

        # Feature projection
        self.feature_proj = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(512, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3)
        )

        # Histogram feature processor - FIXED SIZE
        self.hist_processor = nn.Sequential(
            nn.Linear(histogram_features, 64),  # Uses correct size now
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(64, 64),
            nn.ReLU(inplace=True)
        )

        # Combined classifier
        self.classifier = nn.Sequential(
            nn.Linear(256 + 64, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(128, num_classes)
        )

    def forward(self, images, hist_features):
        # Extract image features
        features = self.encoder(images)
        img_features = self.feature_proj(features[-1])

        # Process histogram features
        hist_out = self.hist_processor(hist_features)

        # Combine and classify
        combined = torch.cat([img_features, hist_out], dim=1)
        output = self.classifier(combined)

        return output

print("✓ HybridHoVerNetGNN redefined with correct histogram size")

# ============================================================
# FIX 4: Define RNN/LSTM Model for comparison
# ============================================================

print("\n[FIX] Defining RNN/LSTM model...")

class HistogramRNN(nn.Module):
    """
    RNN/LSTM model that processes image features sequentially.
    Used for comparison with CNN and HoVerNet models.
    """

    def __init__(self, num_classes=3):
        super(HistogramRNN, self).__init__()

        # CNN feature extractor
        self.cnn = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(4, 4),  # 256 -> 64

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(4, 4),  # 64 -> 16

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((4, 4))  # -> 4x4
        )

        # LSTM for sequential processing
        self.lstm = nn.LSTM(
            input_size=128,
            hidden_size=64,
            num_layers=2,
            batch_first=True,
            dropout=0.3,
            bidirectional=True
        )

        # Classifier
        self.classifier = nn.Sequential(
            nn.Linear(128, 64),  # 64*2 for bidirectional
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        batch_size = x.size(0)

        # CNN features: [B, 128, 4, 4]
        features = self.cnn(x)

        # Reshape for LSTM: treat spatial positions as sequence
        # [B, 128, 4, 4] -> [B, 128, 16] -> [B, 16, 128]
        features = features.view(batch_size, 128, -1)  # [B, 128, 16]
        features = features.permute(0, 2, 1)  # [B, 16, 128]

        # LSTM
        lstm_out, (h_n, c_n) = self.lstm(features)

        # Use final hidden states from both directions
        # h_n shape: [num_layers*2, B, hidden_size] for bidirectional
        out = torch.cat([h_n[-2], h_n[-1]], dim=1)  # [B, 128]

        # Classify
        out = self.classifier(out)
        return out

print("✓ HistogramRNN model defined")

# ============================================================
# START TRAINING ALL MODELS
# ============================================================

print("\n" + "="*60)
print("TRAINING ALL MODELS")
print("="*60)

# Dictionary to store all results
all_results = {}

# ============================================================
# 10.1 Train Simple CNN
# ============================================================
print("\n" + "-"*50)
print("[1/5] Training Simple CNN...")
print("-"*50)

cnn_model = SimpleCNN(num_classes=config.NUM_CLASSES)
cnn_model, cnn_history, cnn_best_acc = train_model(
    cnn_model, train_loader, val_loader, "Simple CNN",
    num_epochs=config.NUM_EPOCHS, learning_rate=config.LEARNING_RATE,
    use_histogram=False
)
all_results['Simple CNN'] = {
    'model': cnn_model, 'history': cnn_history, 'best_val_acc': cnn_best_acc
}

# Clear GPU memory
torch.cuda.empty_cache()
gc.collect()

# ============================================================
# 10.2 Train ResNet50
# ============================================================
print("\n" + "-"*50)
print("[2/5] Training ResNet50...")
print("-"*50)

resnet_model = ResNetClassifier(num_classes=config.NUM_CLASSES, pretrained=True)
resnet_model, resnet_history, resnet_best_acc = train_model(
    resnet_model, train_loader, val_loader, "ResNet50",
    num_epochs=config.NUM_EPOCHS, learning_rate=config.LEARNING_RATE,
    use_histogram=False
)
all_results['ResNet50'] = {
    'model': resnet_model, 'history': resnet_history, 'best_val_acc': resnet_best_acc
}

torch.cuda.empty_cache()
gc.collect()

# ============================================================
# 10.3 Train HoVerNet
# ============================================================
print("\n" + "-"*50)
print("[3/5] Training HoVerNet...")
print("-"*50)

hovernet_model = HoVerNet(num_classes=config.NUM_CLASSES, pretrained=True)
hovernet_model, hovernet_history, hovernet_best_acc = train_model(
    hovernet_model, train_loader, val_loader, "HoVerNet",
    num_epochs=config.NUM_EPOCHS, learning_rate=config.LEARNING_RATE,
    use_histogram=False
)
all_results['HoVerNet'] = {
    'model': hovernet_model, 'history': hovernet_history, 'best_val_acc': hovernet_best_acc
}

torch.cuda.empty_cache()
gc.collect()

# ============================================================
# 10.4 Train Hybrid HoVerNet + GNN (FIXED)
# ============================================================
print("\n" + "-"*50)
print("[4/5] Training Hybrid HoVerNet-GNN...")
print("-"*50)

# Use the ACTUAL histogram size detected from data
hybrid_model = HybridHoVerNetGNN(
    num_classes=config.NUM_CLASSES,
    histogram_features=ACTUAL_HIST_SIZE  # FIXED: Use actual size
)
hybrid_model, hybrid_history, hybrid_best_acc = train_model(
    hybrid_model, train_loader, val_loader, "HoVerNet-GNN",
    num_epochs=config.NUM_EPOCHS, learning_rate=config.LEARNING_RATE,
    use_histogram=True
)
all_results['HoVerNet-GNN'] = {
    'model': hybrid_model, 'history': hybrid_history, 'best_val_acc': hybrid_best_acc
}

torch.cuda.empty_cache()
gc.collect()

# ============================================================
# 10.5 Train RNN/LSTM Model
# ============================================================
print("\n" + "-"*50)
print("[5/5] Training RNN-LSTM...")
print("-"*50)

rnn_model = HistogramRNN(num_classes=config.NUM_CLASSES)
rnn_model, rnn_history, rnn_best_acc = train_model(
    rnn_model, train_loader, val_loader, "RNN-LSTM",
    num_epochs=config.NUM_EPOCHS, learning_rate=config.LEARNING_RATE,
    use_histogram=False  # RNN uses images directly, not histogram features
)
all_results['RNN-LSTM'] = {
    'model': rnn_model, 'history': rnn_history, 'best_val_acc': rnn_best_acc
}

torch.cuda.empty_cache()
gc.collect()

# ============================================================
# TRAINING COMPLETE - SUMMARY
# ============================================================

print("\n" + "="*60)
print("ALL 5 MODELS TRAINED SUCCESSFULLY!")
print("="*60)

print("\n📊 Training Summary:")
print("-" * 50)
print(f"{'Model':<20} {'Best Val Acc':>15}")
print("-" * 50)
for model_name, data in all_results.items():
    print(f"{model_name:<20} {data['best_val_acc']:>14.2f}%")
print("-" * 50)

# Find and highlight best model
best_model_name = max(all_results.keys(), key=lambda x: all_results[x]['best_val_acc'])
best_acc = all_results[best_model_name]['best_val_acc']
print(f"\n🏆 Best Validation Accuracy: {best_model_name} ({best_acc:.2f}%)")

In [ ]:
# ============================================================
# SECTION 11: EVALUATION ON TEST SET
# ============================================================

print("\n" + "="*60)
print("EVALUATING ON TEST SET")
print("="*60)

def evaluate_model(model, test_loader, model_name, use_histogram=False):
    """Evaluate model on test set and return metrics"""

    model.eval()
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()

    _, test_acc, all_preds, all_labels, all_probs = validate_epoch(
        model, test_loader, criterion, device, use_histogram
    )

    # Calculate metrics
    metrics = {
        'accuracy': accuracy_score(all_labels, all_preds),
        'f1_macro': f1_score(all_labels, all_preds, average='macro'),
        'f1_weighted': f1_score(all_labels, all_preds, average='weighted'),
        'precision': precision_score(all_labels, all_preds, average='weighted'),
        'recall': recall_score(all_labels, all_preds, average='weighted'),
        'predictions': all_preds,
        'labels': all_labels,
        'probabilities': all_probs
    }

    print(f"\n{model_name} Test Results:")
    print(f"  Accuracy: {metrics['accuracy']*100:.2f}%")
    print(f"  F1 Score (macro): {metrics['f1_macro']*100:.2f}%")
    print(f"  Precision: {metrics['precision']*100:.2f}%")
    print(f"  Recall: {metrics['recall']*100:.2f}%")

    return metrics

# Evaluate all models
test_results = {}

for model_name, data in all_results.items():
    use_hist = (model_name == 'HoVerNet-GNN')
    metrics = evaluate_model(data['model'], test_loader, model_name, use_histogram=use_hist)
    test_results[model_name] = metrics

In [ ]:
# ============================================================
# SECTION 12: VISUALIZATION AND COMPARISON (COMPLETE FIXED VERSION)
# ============================================================

print("\n" + "="*60)
print("GENERATING VISUALIZATIONS")
print("="*60)

# ============================================================
# 12.1 Training History Comparison
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Training Loss
axes[0, 0].set_title('Training Loss', fontsize=12, fontweight='bold')
for model_name, data in all_results.items():
    axes[0, 0].plot(data['history']['train_loss'], label=model_name, linewidth=2)
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].legend(loc='upper right', fontsize=8)
axes[0, 0].grid(True, alpha=0.3)

# Validation Loss
axes[0, 1].set_title('Validation Loss', fontsize=12, fontweight='bold')
for model_name, data in all_results.items():
    axes[0, 1].plot(data['history']['val_loss'], label=model_name, linewidth=2)
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Loss')
axes[0, 1].legend(loc='upper right', fontsize=8)
axes[0, 1].grid(True, alpha=0.3)

# Training Accuracy
axes[1, 0].set_title('Training Accuracy', fontsize=12, fontweight='bold')
for model_name, data in all_results.items():
    axes[1, 0].plot(data['history']['train_acc'], label=model_name, linewidth=2, marker='o', markersize=3)
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Accuracy (%)')
axes[1, 0].legend(loc='lower right', fontsize=8)
axes[1, 0].grid(True, alpha=0.3)

# Validation Accuracy
axes[1, 1].set_title('Validation Accuracy', fontsize=12, fontweight='bold')
for model_name, data in all_results.items():
    axes[1, 1].plot(data['history']['val_acc'], label=model_name, linewidth=2, marker='o', markersize=3)
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Accuracy (%)')
axes[1, 1].legend(loc='lower right', fontsize=8)
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f"{config.OUTPUT_PATH}/plots/training_history.png", dpi=200, bbox_inches='tight')
plt.show()
print("✓ Training history plot saved")

# ============================================================
# 12.2 Test Accuracy Bar Chart
# ============================================================

fig, ax = plt.subplots(figsize=(12, 6))

model_names = list(test_results.keys())
accuracies = [test_results[m]['accuracy'] * 100 for m in model_names]
colors = ['#3498db', '#e74c3c', '#2ecc71', '#9b59b6', '#f39c12'][:len(model_names)]

bars = ax.bar(range(len(model_names)), accuracies, color=colors, edgecolor='black', linewidth=1.5)

# Add value labels on bars
for bar, acc in zip(bars, accuracies):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + 0.5,
            f'{acc:.2f}%', ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.set_xticks(range(len(model_names)))
ax.set_xticklabels(model_names, rotation=15, ha='right', fontsize=11)
ax.set_ylabel('Test Accuracy (%)', fontsize=12)
ax.set_title('Model Comparison - Test Accuracy', fontsize=14, fontweight='bold')
ax.set_ylim([0, 105])
ax.grid(True, alpha=0.3, axis='y')

# Highlight best model
best_idx = np.argmax(accuracies)
bars[best_idx].set_edgecolor('gold')
bars[best_idx].set_linewidth(3)

plt.tight_layout()
plt.savefig(f"{config.OUTPUT_PATH}/plots/test_accuracy_comparison.png", dpi=200, bbox_inches='tight')
plt.show()
print("✓ Test accuracy comparison plot saved")

# ============================================================
# 12.3 All Metrics Comparison Bar Chart
# ============================================================

metrics_names = ['accuracy', 'f1_macro', 'precision', 'recall']
metrics_labels = ['Accuracy', 'F1 Score', 'Precision', 'Recall']

fig, ax = plt.subplots(figsize=(14, 6))

x = np.arange(len(model_names))
width = 0.18
multiplier = 0

colors_metrics = ['#3498db', '#2ecc71', '#f39c12', '#e74c3c']

for metric, label, color in zip(metrics_names, metrics_labels, colors_metrics):
    values = [test_results[m][metric] * 100 for m in model_names]
    offset = width * multiplier
    bars = ax.bar(x + offset, values, width, label=label, color=color, edgecolor='black')
    multiplier += 1

ax.set_ylabel('Score (%)', fontsize=12)
ax.set_title('All Performance Metrics Comparison', fontsize=14, fontweight='bold')
ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(model_names, rotation=15, ha='right', fontsize=10)
ax.legend(loc='lower right', fontsize=10)
ax.set_ylim([0, 105])
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(f"{config.OUTPUT_PATH}/plots/all_metrics_comparison.png", dpi=200, bbox_inches='tight')
plt.show()
print("✓ All metrics comparison plot saved")

# ============================================================
# 12.4 Confusion Matrices for All Models (FIXED FOR 5 MODELS)
# ============================================================

n_models = len(test_results)
n_cols = 3
n_rows = 2

fig, axes = plt.subplots(n_rows, n_cols, figsize=(5*n_cols, 5*n_rows))
axes = axes.flatten()

for idx, (model_name, metrics) in enumerate(test_results.items()):
    cm = confusion_matrix(metrics['labels'], metrics['predictions'])

    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx],
                xticklabels=class_labels, yticklabels=class_labels,
                annot_kws={'size': 12})
    axes[idx].set_title(f'{model_name}\nAccuracy: {metrics["accuracy"]*100:.2f}%',
                        fontsize=11, fontweight='bold')
    axes[idx].set_xlabel('Predicted', fontsize=10)
    axes[idx].set_ylabel('Actual', fontsize=10)

# Hide unused subplots
for idx in range(n_models, len(axes)):
    axes[idx].axis('off')

plt.suptitle('Confusion Matrices - All Models', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f"{config.OUTPUT_PATH}/plots/confusion_matrices.png", dpi=200, bbox_inches='tight')
plt.show()
print("✓ Confusion matrices saved")

# ============================================================
# 12.5 ROC Curves for All Models (FIXED - NaN HANDLING)
# ============================================================

print("Generating ROC curves...")

# Color mapping for models
colors_models = {
    'Simple CNN': '#3498db',
    'ResNet50': '#e74c3c',
    'HoVerNet': '#2ecc71',
    'HoVerNet-GNN': '#9b59b6',
    'RNN-LSTM': '#f39c12'
}

fig, axes = plt.subplots(1, config.NUM_CLASSES, figsize=(6*config.NUM_CLASSES, 5))

# Handle case where NUM_CLASSES might be 1
if config.NUM_CLASSES == 1:
    axes = [axes]

# Store AUC values for later use
auc_scores = {model_name: [] for model_name in test_results.keys()}

for class_idx in range(config.NUM_CLASSES):
    ax = axes[class_idx]

    for model_name, metrics in test_results.items():
        try:
            # Binarize for current class
            y_true = (metrics['labels'] == class_idx).astype(int)
            y_score = metrics['probabilities'][:, class_idx].copy()

            # FIX: Handle NaN values
            # Replace NaN with 0 (or you could use mean of non-NaN values)
            nan_mask = np.isnan(y_score)
            if nan_mask.any():
                print(f"  Warning: {model_name} has {nan_mask.sum()} NaN values in class {class_idx} probabilities. Replacing with 0.")
                y_score[nan_mask] = 0.0

            # Also handle Inf values
            inf_mask = np.isinf(y_score)
            if inf_mask.any():
                print(f"  Warning: {model_name} has {inf_mask.sum()} Inf values. Replacing.")
                y_score[inf_mask] = 1.0 if y_score[inf_mask].mean() > 0 else 0.0

            # Check if we have valid data
            if len(np.unique(y_true)) < 2:
                print(f"  Warning: {model_name} class {class_idx} has only one class in y_true. Skipping ROC.")
                auc_scores[model_name].append(0.5)
                continue

            fpr, tpr, _ = roc_curve(y_true, y_score)
            roc_auc = auc(fpr, tpr)
            auc_scores[model_name].append(roc_auc)

            color = colors_models.get(model_name, '#666666')
            linewidth = 3 if 'HoVerNet' in model_name else 2
            linestyle = '-' if 'HoVerNet' in model_name else '--'

            ax.plot(fpr, tpr, label=f'{model_name} (AUC={roc_auc:.3f})',
                    color=color, linewidth=linewidth, linestyle=linestyle)

        except Exception as e:
            print(f"  Error computing ROC for {model_name}, class {class_idx}: {e}")
            auc_scores[model_name].append(0.5)  # Default AUC for failed cases

    # Random classifier line
    ax.plot([0, 1], [0, 1], 'k--', linewidth=1, alpha=0.5, label='Random')

    ax.set_xlabel('False Positive Rate', fontsize=10)
    ax.set_ylabel('True Positive Rate', fontsize=10)
    ax.set_title(f'ROC Curve - {class_labels[class_idx]}', fontsize=11, fontweight='bold')
    ax.legend(loc='lower right', fontsize=8)
    ax.grid(True, alpha=0.3)
    ax.set_xlim([0, 1])
    ax.set_ylim([0, 1.05])

plt.suptitle('ROC Curves - All Models', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f"{config.OUTPUT_PATH}/plots/roc_curves.png", dpi=200, bbox_inches='tight')
plt.show()
print("✓ ROC curves saved")

# ============================================================
# 12.6 Precision-Recall Curves (FIXED - NaN HANDLING)
# ============================================================

print("Generating Precision-Recall curves...")

fig, axes = plt.subplots(1, config.NUM_CLASSES, figsize=(6*config.NUM_CLASSES, 5))

if config.NUM_CLASSES == 1:
    axes = [axes]

pr_auc_scores = {model_name: [] for model_name in test_results.keys()}

for class_idx in range(config.NUM_CLASSES):
    ax = axes[class_idx]

    for model_name, metrics in test_results.items():
        try:
            y_true = (metrics['labels'] == class_idx).astype(int)
            y_score = metrics['probabilities'][:, class_idx].copy()

            # FIX: Handle NaN values
            nan_mask = np.isnan(y_score)
            if nan_mask.any():
                y_score[nan_mask] = 0.0

            # Handle Inf values
            inf_mask = np.isinf(y_score)
            if inf_mask.any():
                y_score[inf_mask] = 1.0 if y_score[inf_mask].mean() > 0 else 0.0

            # Check if we have valid data
            if len(np.unique(y_true)) < 2:
                pr_auc_scores[model_name].append(0.5)
                continue

            precision, recall, _ = precision_recall_curve(y_true, y_score)
            pr_auc = auc(recall, precision)
            pr_auc_scores[model_name].append(pr_auc)

            color = colors_models.get(model_name, '#666666')
            linewidth = 3 if 'HoVerNet' in model_name else 2

            ax.plot(recall, precision, label=f'{model_name} (AUC={pr_auc:.3f})',
                    color=color, linewidth=linewidth)

        except Exception as e:
            print(f"  Error computing PR for {model_name}, class {class_idx}: {e}")
            pr_auc_scores[model_name].append(0.5)

    ax.set_xlabel('Recall', fontsize=10)
    ax.set_ylabel('Precision', fontsize=10)
    ax.set_title(f'Precision-Recall - {class_labels[class_idx]}', fontsize=11, fontweight='bold')
    ax.legend(loc='lower left', fontsize=8)
    ax.grid(True, alpha=0.3)
    ax.set_xlim([0, 1])
    ax.set_ylim([0, 1.05])

plt.suptitle('Precision-Recall Curves - All Models', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f"{config.OUTPUT_PATH}/plots/precision_recall_curves.png", dpi=200, bbox_inches='tight')
plt.show()
print("✓ Precision-Recall curves saved")

# ============================================================
# 12.7 Summary Table Visualization
# ============================================================

fig, ax = plt.subplots(figsize=(12, 4))
ax.axis('off')

# Create data for table
table_data = []
for model_name in model_names:
    metrics = test_results[model_name]
    table_data.append([
        model_name,
        f"{metrics['accuracy']*100:.2f}%",
        f"{metrics['f1_macro']*100:.2f}%",
        f"{metrics['precision']*100:.2f}%",
        f"{metrics['recall']*100:.2f}%"
    ])

# Find best model for highlighting
best_idx = np.argmax([test_results[m]['accuracy'] for m in model_names])

# Create table
table = ax.table(
    cellText=table_data,
    colLabels=['Model', 'Accuracy', 'F1 Score', 'Precision', 'Recall'],
    cellLoc='center',
    loc='center',
    colColours=['#4a90d9'] * 5
)

# Style the table
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1.2, 1.8)

# Highlight header
for i in range(5):
    table[(0, i)].set_text_props(color='white', fontweight='bold')

# Highlight best model row
for i in range(5):
    table[(best_idx + 1, i)].set_facecolor('#d4edda')

plt.title('Model Performance Summary', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig(f"{config.OUTPUT_PATH}/plots/performance_summary_table.png", dpi=200, bbox_inches='tight')
plt.show()
print("✓ Performance summary table saved")

# ============================================================
# 12.8 Combined AUC Comparison Bar Chart (FIXED)
# ============================================================

fig, ax = plt.subplots(figsize=(12, 6))

# Calculate mean AUC across all classes for each model (use pre-computed values)
mean_aucs = []
for model_name in model_names:
    if model_name in auc_scores and len(auc_scores[model_name]) > 0:
        mean_auc = np.mean(auc_scores[model_name])
    else:
        mean_auc = 0.5  # Default
    mean_aucs.append(mean_auc)

# Plot
bars = ax.bar(range(len(model_names)), mean_aucs, color=colors[:len(model_names)],
              edgecolor='black', linewidth=1.5)

# Add value labels
for bar, auc_val in zip(bars, mean_aucs):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + 0.01,
            f'{auc_val:.3f}', ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.set_xticks(range(len(model_names)))
ax.set_xticklabels(model_names, rotation=15, ha='right', fontsize=11)
ax.set_ylabel('Mean AUC Score', fontsize=12)
ax.set_title('Mean AUC Comparison Across All Classes', fontsize=14, fontweight='bold')
ax.set_ylim([0, 1.1])
ax.grid(True, alpha=0.3, axis='y')

# Highlight best
best_auc_idx = np.argmax(mean_aucs)
bars[best_auc_idx].set_edgecolor('gold')
bars[best_auc_idx].set_linewidth(3)

plt.tight_layout()
plt.savefig(f"{config.OUTPUT_PATH}/plots/mean_auc_comparison.png", dpi=200, bbox_inches='tight')
plt.show()
print("✓ Mean AUC comparison saved")

# ============================================================
# 12.9 Print Summary Statistics
# ============================================================

print("\n" + "="*60)
print("VISUALIZATION SUMMARY")
print("="*60)

print("\n📊 Mean AUC Scores per Model:")
print("-" * 40)
for model_name, m_auc in zip(model_names, mean_aucs):
    print(f"  {model_name:<20}: {m_auc:.4f}")

print("\n" + "="*60)
print("ALL VISUALIZATIONS GENERATED SUCCESSFULLY!")
print("="*60)
print(f"\nAll plots saved to: {config.OUTPUT_PATH}/plots/")

In [ ]:
# ============================================================
# SECTION 13: EXPLAINABILITY HEATMAPS (XAI)
# ============================================================

print("\n" + "="*60)
print("GENERATING EXPLAINABILITY HEATMAPS")
print("="*60)

class GradCAM:
    """Grad-CAM for explainability visualization"""

    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None

        # Register hooks
        target_layer.register_forward_hook(self.save_activation)
        target_layer.register_full_backward_hook(self.save_gradient)

    def save_activation(self, module, input, output):
        self.activations = output.detach()

    def save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()

    def generate(self, input_image, target_class=None):
        self.model.eval()

        # Forward pass
        output = self.model(input_image)
        if isinstance(output, dict):
            output = output['class']

        if target_class is None:
            target_class = output.argmax(dim=1)

        # Backward pass
        self.model.zero_grad()
        one_hot = torch.zeros_like(output)
        one_hot[0, target_class] = 1
        output.backward(gradient=one_hot, retain_graph=True)

        # Generate heatmap
        weights = self.gradients.mean(dim=(2, 3), keepdim=True)
        cam = (weights * self.activations).sum(dim=1, keepdim=True)
        cam = F.relu(cam)
        cam = F.interpolate(cam, size=input_image.shape[2:], mode='bilinear', align_corners=False)
        cam = cam - cam.min()
        cam = cam / (cam.max() + 1e-8)

        return cam.squeeze().cpu().numpy()

# Generate sample heatmaps
def generate_sample_heatmaps(model, dataset, model_name, num_samples=6):
    """Generate Grad-CAM heatmaps for sample images"""

    model = model.to(device)
    model.eval()

    # Get target layer based on model type
    if hasattr(model, 'resnet'):
        target_layer = model.resnet.layer4[-1]
    elif hasattr(model, 'encoder'):
        target_layer = model.encoder.layer4
    else:
        target_layer = list(model.features.children())[-3]

    try:
        grad_cam = GradCAM(model, target_layer)
    except Exception as e:
        print(f"Could not create GradCAM for {model_name}: {e}")
        return

    fig, axes = plt.subplots(2, num_samples, figsize=(3*num_samples, 6))

    indices = np.random.choice(len(dataset), num_samples, replace=False)

    for idx, sample_idx in enumerate(indices):
        # Get sample
        if len(dataset[sample_idx]) == 3:
            image, label, _ = dataset[sample_idx]
        else:
            image, label = dataset[sample_idx]

        input_tensor = image.unsqueeze(0).to(device)

        # Generate heatmap
        try:
            heatmap = grad_cam.generate(input_tensor, label)
        except Exception as e:
            print(f"Error generating heatmap: {e}")
            continue

        # Denormalize image for display
        img_display = image.permute(1, 2, 0).numpy()
        img_display = img_display * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
        img_display = np.clip(img_display, 0, 1)

        # Original image
        axes[0, idx].imshow(img_display)
        axes[0, idx].set_title(f'True: {class_labels[label]}')
        axes[0, idx].axis('off')

        # Heatmap overlay
        axes[1, idx].imshow(img_display)
        axes[1, idx].imshow(heatmap, cmap='jet', alpha=0.5)
        axes[1, idx].set_title('Grad-CAM')
        axes[1, idx].axis('off')

    plt.suptitle(f'Explainability Heatmaps - {model_name}', fontsize=14)
    plt.tight_layout()
    plt.savefig(f"{config.OUTPUT_PATH}/plots/gradcam_{model_name.lower().replace(' ', '_')}.png",
                dpi=200, bbox_inches='tight')
    plt.show()

# Generate heatmaps for best models
for model_name in ['ResNet50', 'HoVerNet']:
    if model_name in all_results:
        try:
            generate_sample_heatmaps(
                all_results[model_name]['model'],
                test_dataset,
                model_name
            )
        except Exception as e:
            print(f"Could not generate heatmaps for {model_name}: {e}")

In [ ]:
# ============================================================
# SECTION 14: NUCLEUS SEGMENTATION VISUALIZATION
# ============================================================

print("\n" + "="*60)
print("NUCLEUS SEGMENTATION EXAMPLES")
print("="*60)

def visualize_segmentation(model, dataset, num_samples=4):
    """Visualize nucleus probability maps from HoVerNet"""

    model = model.to(device)
    model.eval()

    fig, axes = plt.subplots(3, num_samples, figsize=(4*num_samples, 10))

    indices = np.random.choice(len(dataset), num_samples, replace=False)

    for idx, sample_idx in enumerate(indices):
        if len(dataset[sample_idx]) == 3:
            image, label, _ = dataset[sample_idx]
        else:
            image, label = dataset[sample_idx]

        input_tensor = image.unsqueeze(0).to(device)

        with torch.no_grad():
            output = model(input_tensor)

        # Denormalize image
        img_display = image.permute(1, 2, 0).numpy()
        img_display = img_display * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
        img_display = np.clip(img_display, 0, 1)

        # Original image
        axes[0, idx].imshow(img_display)
        axes[0, idx].set_title(f'Original ({class_labels[label]})')
        axes[0, idx].axis('off')

        # Nucleus probability map
        np_map = output['np'][0, 0].cpu().numpy()
        axes[1, idx].imshow(np_map, cmap='hot')
        axes[1, idx].set_title('Nucleus Probability')
        axes[1, idx].axis('off')

        # HV gradients
        hv_map = output['hv'][0].cpu().numpy()
        hv_magnitude = np.sqrt(hv_map[0]**2 + hv_map[1]**2)
        axes[2, idx].imshow(hv_magnitude, cmap='viridis')
        axes[2, idx].set_title('HV Gradient Magnitude')
        axes[2, idx].axis('off')

    plt.suptitle('HoVerNet Segmentation Output', fontsize=14)
    plt.tight_layout()
    plt.savefig(f"{config.OUTPUT_PATH}/plots/hovernet_segmentation.png", dpi=200, bbox_inches='tight')
    plt.show()

# Visualize HoVerNet segmentation
if 'HoVerNet' in all_results:
    visualize_segmentation(all_results['HoVerNet']['model'], test_dataset)

In [ ]:
# ============================================================
# SECTION 14.5: SPATIAL RELATIONSHIP VISUALIZATION
# ============================================================

print("\n" + "="*60)
print("GENERATING SPATIAL RELATIONSHIP MAPS")
print("="*60)

def visualize_complete_analysis(test_df, hovernet_model, num_samples=4):
    """
    Complete visualization including:
    - Original image
    - Nuclei detection
    - Spatial relationship graph
    - Cell density heatmap
    - HoVerNet segmentation
    """

    fig, axes = plt.subplots(5, num_samples, figsize=(4*num_samples, 18))

    sample_indices = np.random.choice(len(test_df), num_samples, replace=False)

    for idx, sample_idx in enumerate(sample_indices):
        row = test_df.iloc[sample_idx]
        image_path = row['image_path']
        true_label = row['label']

        # Load image
        image = cv2.imread(image_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = cv2.resize(image, (256, 256))

        # Detect nuclei
        gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
        binary = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                       cv2.THRESH_BINARY_INV, 21, 5)
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
        binary = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel, iterations=2)

        contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        nuclei = []
        for contour in contours:
            area = cv2.contourArea(contour)
            if 50 < area < 5000:
                M = cv2.moments(contour)
                if M["m00"] > 0:
                    cx = int(M["m10"] / M["m00"])
                    cy = int(M["m01"] / M["m00"])
                    nuclei.append({
                        'centroid': (cx, cy),
                        'area': area
                    })

        # Build edges
        if len(nuclei) >= 2:
            centroids = np.array([n['centroid'] for n in nuclei])
            from scipy.spatial.distance import cdist
            distances = cdist(centroids, centroids)
            edges = []
            for i in range(len(nuclei)):
                sorted_idx = np.argsort(distances[i])
                for j in sorted_idx[1:5]:
                    if distances[i, j] < 80:
                        edges.append([i, j])
            edge_array = np.array(edges).T if edges else np.array([[],[]])
        else:
            edge_array = np.array([[],[]])

        # Row 0: Original Image
        axes[0, idx].imshow(image)
        axes[0, idx].set_title(f'Original\n({class_labels[true_label]})')
        axes[0, idx].axis('off')

        # Row 1: Nuclei Detection
        nuclei_img = image.copy()
        for nucleus in nuclei:
            center = tuple(map(int, nucleus['centroid']))
            radius = int(np.sqrt(nucleus['area'] / np.pi))
            cv2.circle(nuclei_img, center, radius, (255, 0, 0), 2)
        axes[1, idx].imshow(nuclei_img)
        axes[1, idx].set_title(f'Nuclei Detection\n({len(nuclei)} cells)')
        axes[1, idx].axis('off')

        # Row 2: Spatial Relationship Graph
        graph_img = image.copy()
        if edge_array.size > 0:
            for i in range(edge_array.shape[1]):
                src, dst = edge_array[:, i]
                if src < len(nuclei) and dst < len(nuclei):
                    pt1 = tuple(map(int, nuclei[src]['centroid']))
                    pt2 = tuple(map(int, nuclei[dst]['centroid']))
                    cv2.line(graph_img, pt1, pt2, (0, 255, 255), 1)
        for nucleus in nuclei:
            center = tuple(map(int, nucleus['centroid']))
            cv2.circle(graph_img, center, 4, (0, 255, 0), -1)
        axes[2, idx].imshow(graph_img)
        axes[2, idx].set_title(f'Spatial Graph\n({edge_array.shape[1] if edge_array.size > 0 else 0} edges)')
        axes[2, idx].axis('off')

        # Row 3: Cell Density Heatmap
        density_map, density_colored = create_cell_density_heatmap(image.shape, nuclei)
        axes[3, idx].imshow(density_colored)
        axes[3, idx].set_title(f'Cell Density Map')
        axes[3, idx].axis('off')

        # Row 4: HoVerNet Segmentation (if available)
        hovernet_model.eval()
        transform = transforms.Compose([
            transforms.ToPILImage(),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
        input_tensor = transform(image).unsqueeze(0).to(device)

        with torch.no_grad():
            output = hovernet_model(input_tensor)

        np_map = output['np'][0, 0].cpu().numpy()
        axes[4, idx].imshow(np_map, cmap='hot')
        axes[4, idx].set_title('HoVerNet Output')
        axes[4, idx].axis('off')

    plt.suptitle('Complete Analysis Pipeline', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f"{config.OUTPUT_PATH}/plots/complete_spatial_analysis.png", dpi=200, bbox_inches='tight')
    plt.show()

# Generate complete visualization
if 'HoVerNet' in all_results:
    visualize_complete_analysis(test_df, all_results['HoVerNet']['model'])

In [ ]:
# ============================================================
# SECTION 15: TUMOR SIZE ESTIMATION
# ============================================================

print("\n" + "="*60)
print("TUMOR SIZE ESTIMATION")
print("="*60)

def estimate_tumor_size(model, image_path, threshold=0.5):
    """
    Estimate tumor size based on nucleus segmentation.
    Returns: tumor area, cell count, density metrics
    """
    model = model.to(device)
    model.eval()

    # Load and preprocess image
    image = cv2.imread(image_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    original_size = image.shape[:2]

    # Resize for model
    image_resized = cv2.resize(image, (config.IMAGE_SIZE, config.IMAGE_SIZE))

    # Transform
    transform = transforms.Compose([
        transforms.ToPILImage(),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    input_tensor = transform(image_resized).unsqueeze(0).to(device)

    with torch.no_grad():
        output = model(input_tensor)

    # Get nucleus probability map
    np_map = output['np'][0, 0].cpu().numpy()

    # Threshold to get nuclei regions
    nuclei_mask = (np_map > threshold).astype(np.uint8)

    # Count nuclei using connected components
    labeled_array, num_nuclei = scipy_label(nuclei_mask)

    # Calculate metrics
    tumor_pixel_area = np.sum(nuclei_mask)
    tumor_percentage = (tumor_pixel_area / nuclei_mask.size) * 100

    # Scale to original image size
    scale_factor = (original_size[0] / config.IMAGE_SIZE) * (original_size[1] / config.IMAGE_SIZE)
    estimated_actual_area = tumor_pixel_area * scale_factor

    # Get predicted class
    class_probs = F.softmax(output['class'], dim=1)[0].cpu().numpy()
    predicted_class = np.argmax(class_probs)

    return {
        'num_nuclei': num_nuclei,
        'tumor_pixel_area': tumor_pixel_area,
        'tumor_percentage': tumor_percentage,
        'estimated_actual_area': estimated_actual_area,
        'predicted_class': predicted_class,
        'class_probabilities': class_probs,
        'np_map': np_map,
        'nuclei_mask': nuclei_mask
    }

# Demo tumor size estimation
def demo_tumor_estimation(model, test_df, num_samples=4):
    """Demonstrate tumor size estimation on sample images"""

    fig, axes = plt.subplots(3, num_samples, figsize=(4*num_samples, 10))

    sample_indices = np.random.choice(len(test_df), num_samples, replace=False)

    for idx, sample_idx in enumerate(sample_indices):
        image_path = test_df.iloc[sample_idx]['image_path']
        true_label = test_df.iloc[sample_idx]['label']
        cell_index = test_df.iloc[sample_idx]['cell_index']

        # Estimate tumor size
        results = estimate_tumor_size(model, image_path)

        # Load original image
        image = cv2.imread(image_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = cv2.resize(image, (config.IMAGE_SIZE, config.IMAGE_SIZE))

        # Original image
        axes[0, idx].imshow(image)
        axes[0, idx].set_title(f'True: {class_labels[true_label]}\nCell Index: {cell_index:.2f}')
        axes[0, idx].axis('off')

        # Probability map
        axes[1, idx].imshow(results['np_map'], cmap='hot')
        axes[1, idx].set_title(f'Nuclei: {results["num_nuclei"]}\nArea: {results["tumor_percentage"]:.1f}%')
        axes[1, idx].axis('off')

        # Overlay
        axes[2, idx].imshow(image)
        axes[2, idx].imshow(results['nuclei_mask'], cmap='Reds', alpha=0.5)
        pred_class = class_labels[results['predicted_class']]
        axes[2, idx].set_title(f'Pred: {pred_class}\nConf: {results["class_probabilities"].max()*100:.1f}%')
        axes[2, idx].axis('off')

    plt.suptitle('Tumor Size Estimation Examples', fontsize=14)
    plt.tight_layout()
    plt.savefig(f"{config.OUTPUT_PATH}/plots/tumor_size_estimation.png", dpi=200, bbox_inches='tight')
    plt.show()

if 'HoVerNet' in all_results:
    demo_tumor_estimation(all_results['HoVerNet']['model'], test_df)

In [ ]:
# ============================================================
# SECTION 16: SAVE MODELS AND RESULTS
# ============================================================

print("\n" + "="*60)
print("SAVING MODELS AND RESULTS")
print("="*60)

# Save models
for model_name, data in all_results.items():
    model_path = f"{config.OUTPUT_PATH}/models/{model_name.lower().replace(' ', '_').replace('-', '_')}_model.pth"
    torch.save({
        'model_state_dict': data['model'].state_dict(),
        'history': data['history'],
        'best_val_acc': data['best_val_acc'],
        'config': vars(config)
    }, model_path)
    print(f"Saved: {model_path}")

# Save results summary
results_summary = {
    'class_labels': class_labels,
    'models': {}
}

for model_name, metrics in test_results.items():
    results_summary['models'][model_name] = {
        'accuracy': float(metrics['accuracy']),
        'f1_macro': float(metrics['f1_macro']),
        'f1_weighted': float(metrics['f1_weighted']),
        'precision': float(metrics['precision']),
        'recall': float(metrics['recall'])
    }

with open(f"{config.OUTPUT_PATH}/results/test_results.json", 'w') as f:
    json.dump(results_summary, f, indent=2)

print(f"\nResults saved to: {config.OUTPUT_PATH}/results/test_results.json")

In [ ]:
# ============================================================
# SECTION 17: FINAL SUMMARY
# ============================================================

print("\n" + "="*60)
print("FINAL SUMMARY")
print("="*60)

# Create summary table
summary_df = pd.DataFrame({
    'Model': list(test_results.keys()),
    'Accuracy (%)': [f"{test_results[m]['accuracy']*100:.2f}" for m in test_results],
    'F1 Score (%)': [f"{test_results[m]['f1_macro']*100:.2f}" for m in test_results],
    'Precision (%)': [f"{test_results[m]['precision']*100:.2f}" for m in test_results],
    'Recall (%)': [f"{test_results[m]['recall']*100:.2f}" for m in test_results]
})

print("\n" + summary_df.to_string(index=False))

# Find best model
best_model_name = max(test_results, key=lambda x: test_results[x]['accuracy'])
print(f"\n🏆 BEST MODEL: {best_model_name} with {test_results[best_model_name]['accuracy']*100:.2f}% accuracy")

# Save summary
summary_df.to_csv(f"{config.OUTPUT_PATH}/results/model_comparison.csv", index=False)

print("\n" + "="*60)
print("TRAINING COMPLETE!")
print("="*60)
print(f"\nAll outputs saved to: {config.OUTPUT_PATH}")
print("\nGenerated files:")
print("  - models/: Trained model weights")
print("  - plots/: All visualizations")
print("  - results/: Metrics and comparisons")

# List all generated files
print("\n📁 Generated files:")
for root, dirs, files in os.walk(config.OUTPUT_PATH):
    for file in files:
        filepath = os.path.join(root, file)
        size = os.path.getsize(filepath) / 1024
        print(f"  - {filepath.replace(config.OUTPUT_PATH, '')} ({size:.1f} KB)")


In [ ]:
# ============================================================
# CONFUSION MATRICES FOR ALL 5 MODELS
# ============================================================

num_models = len(test_results)
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for idx, (model_name, metrics) in enumerate(test_results.items()):
    if idx < 6:
        cm = confusion_matrix(metrics['labels'], metrics['predictions'])
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx],
                    xticklabels=class_labels, yticklabels=class_labels)
        axes[idx].set_title(f'{model_name}\nAccuracy: {metrics["accuracy"]*100:.2f}%')
        axes[idx].set_xlabel('Predicted')
        axes[idx].set_ylabel('Actual')

# Hide unused subplot
if num_models < 6:
    axes[5].axis('off')

plt.suptitle('Confusion Matrices - All Models', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f"{config.OUTPUT_PATH}/plots/all_confusion_matrices.png", dpi=200, bbox_inches='tight')
plt.show()


# ============================================================
# COMPREHENSIVE ROC CURVES (ALL MODELS, ALL CLASSES)
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

colors_models = {
    'Simple CNN': '#3498db',
    'ResNet50': '#e74c3c',
    'HoVerNet': '#2ecc71',
    'HoVerNet-GNN': '#9b59b6',
    'RNN-LSTM': '#f39c12'
}

for class_idx in range(config.NUM_CLASSES):
    ax = axes[class_idx]

    for model_name, metrics in test_results.items():
        y_true = (metrics['labels'] == class_idx).astype(int)
        y_score = metrics['probabilities'][:, class_idx]

        fpr, tpr, _ = roc_curve(y_true, y_score)
        roc_auc = auc(fpr, tpr)

        color = colors_models.get(model_name, '#666')
        linewidth = 3 if 'HoVerNet' in model_name else 2
        ax.plot(fpr, tpr, label=f'{model_name} (AUC={roc_auc:.3f})',
                color=color, linewidth=linewidth)

    ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random')
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.set_title(f'ROC Curve - {class_labels[class_idx]}')
    ax.legend(loc='lower right', fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle('ROC Curves Comparison - All Models', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f"{config.OUTPUT_PATH}/plots/comprehensive_roc_curves.png", dpi=200, bbox_inches='tight')
plt.show()


# ============================================================
# FINAL SUMMARY TABLE
# ============================================================

print("\n" + "="*70)
print("FINAL COMPREHENSIVE SUMMARY")
print("="*70)

summary_data = []
for model_name, metrics in test_results.items():
    summary_data.append({
        'Model': model_name,
        'Accuracy (%)': f"{metrics['accuracy']*100:.2f}",
        'F1 Macro (%)': f"{metrics['f1_macro']*100:.2f}",
        'F1 Weighted (%)': f"{metrics['f1_weighted']*100:.2f}",
        'Precision (%)': f"{metrics['precision']*100:.2f}",
        'Recall (%)': f"{metrics['recall']*100:.2f}"
    })

summary_df = pd.DataFrame(summary_data)
print("\n" + summary_df.to_string(index=False))

# Highlight best model
best_model = max(test_results.items(), key=lambda x: x[1]['accuracy'])
print(f"\n{'='*70}")
print(f"🏆 BEST PERFORMING MODEL: {best_model[0]}")
print(f"   Accuracy: {best_model[1]['accuracy']*100:.2f}%")
print(f"   F1 Score: {best_model[1]['f1_macro']*100:.2f}%")
print(f"{'='*70}")

# Save final summary
summary_df.to_csv(f"{config.OUTPUT_PATH}/results/final_comparison.csv", index=False)